# Section 7.2 -- Real-World Tools for Agents
## Your Own APIs, Postman, Live APIs (Travel, Outlook, Zoom, Slack) and SQL Databases

**Author:** Suresh D R | AI Product Developer & Technology Mentor

---

**Framework:** CrewAI

In Section 7 our tools returned **mock data**. Real agents connect to **real systems**.
This notebook builds that skill **one layer at a time**:

| Part | What You Build | Why This Order |
|---|---|---|
| **A** | Your **first API** (2 endpoints) → test in **Postman from Colab** → wrap as tools → give to an agent | The whole agent ↔ API loop in its simplest form |
| **B** | A **real API** with a database, an API key and more endpoints → Postman → agent | Same loop, closer to a company API |
| **C** | **Real-time external APIs** -- weather, currency, flights (Duffel), **Outlook**, **Zoom**, **Slack** | Same loop, someone else's API -- add **authentication** |
| **D** | A **Text-to-SQL agent** on the database | A different tool: the agent **writes the query itself** |
| **E** | One **Operations Agent** using SQL + Zoom + Outlook + Slack | Everything combined |

**Company: DV Analytics -- Data Science and AI Training Institute (EdTech)**

| Step | What You Build |
|---|---|
| | **PART A -- Your First API** |
| 3 | Build the Fee API with Flask (2 endpoints) |
| 4 | Run it and test it inside Colab |
| 5 | Test it from **Postman** (Colab → ngrok → Postman) |
| 6 | Wrap the API as tools |
| 7 | Give the tools to an agent -- the WhatsApp fee question |
| | **PART B -- A Real API with a Database** |
| 8 | Create the DV Analytics database |
| 9 | Build the Admissions API (database, API key, 5 endpoints) |
| 10 | Run it and test every endpoint with `requests` |
| 11 | Test from Postman + import a ready Postman collection |
| 12 | Give the API to an agent -- register a lead |
| | **PART C -- Real-Time External APIs** |
| 13 | How HTTP API tools work -- method, URL, headers, auth, body, status codes |
| 14 | Live free APIs -- weather and currency (no key) |
| 15 | Flight search API -- Duffel (why not MakeMyTrip) + Travel Agent |
| 16 | Outlook tools -- Microsoft Graph (email + calendar) |
| 17 | Zoom tool -- create meetings |
| 18 | Slack tool -- post team updates |
| | **PART D -- Agents on a SQL Database** |
| 19 | SQL tools -- read the schema, run **read-only** queries |
| 20 | SQL Analyst Agent -- plain English → SQL → answer |
| 21 | Safety test -- the agent tries to DELETE data and is blocked |
| | **PART E -- Putting It All Together** |
| 22 | Operations Agent -- SQL + Zoom + Outlook + Slack in one task |

> **Credentials:** Steps 15-18 work **without any keys** in **DRY RUN** mode. Postman from Colab (Steps 5 and 11) needs a free **ngrok** token.

---
## How an Agent and an API Talk to Each Other

**Real-time example:** at 9 PM a student messages DV Analytics on WhatsApp:

> *"What is the fee for the Agentic AI course, and how much per month if I pay in 6 EMIs?"*

The fee is **not** in the LLM's memory -- it lives in the company's system, behind an **API**.

```
 Student                Agent (LLM)              Tool (Python)              API (Flask)
    │  "fee + EMI?"         │                         │                         │
    │──────────────────────>│                         │                         │
    │                       │ 1. decides: I need the  │                         │
    │                       │    fee → call the tool  │                         │
    │                       │────────────────────────>│                         │
    │                       │                         │ 2. GET /fee?course=...  │
    │                       │                         │────────────────────────>│
    │                       │                         │ 3. {"fee": 74999}       │
    │                       │                         │<────────────────────────│
    │                       │ 4. tool returns the     │                         │
    │                       │    JSON as text         │                         │
    │                       │<────────────────────────│                         │
    │                       │ 5. decides: now the EMI → POST /emi → {"monthly_emi": 12500}
    │  6. "Rs 74,999, or    │                         │                         │
    │   Rs 12,500 x 6"      │                         │                         │
    │<──────────────────────│                         │                         │
```

**Who does what:**

| Part | Its job | In this notebook |
|---|---|---|
| **Agent (LLM)** | Understands the question, **decides** which tool to call and with what input | CrewAI Agent + GPT-4o-mini |
| **Tool** | A Python function that turns the agent's decision into an **HTTP request** | `@tool` + `requests` |
| **API** | Receives the request, does the work, returns **JSON** | Flask |

**The key point:** the LLM **never** calls the API itself. It only *chooses* a tool and its input. **Your tool code** makes the HTTP call.

**Analogy -- a restaurant:** the student is the **customer**, the agent is the **waiter**, the tool is the **order slip**, and the API is the **kitchen counter**. The waiter never cooks -- he writes the order on a slip and passes it to the kitchen.

In **Part A** we build exactly this example, in 5 small steps.

---
## Step 1 -- Install

In [1]:
# Step 1 — Run this cell first. Then Runtime > Restart Session. Then run from Step 2.
!pip install -q crewai==1.14.3 pydantic==2.11.9 "openai>=2.0.0" flask requests pandas pyngrok
print("Done! Go to Runtime > Restart Session.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 28.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.9/444.9 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 71.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.9/19.9 MB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.1/260.1 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 353

---
## Step 2 -- Setup

**Never type your API key into a notebook.** Anyone you share the notebook with can use it and bill your account.

**OpenAI key -- 3 ways (tried in this order):**
1. **Colab Secrets (recommended):** key icon in the left sidebar → add `OPENAI_API_KEY` → turn on *Notebook access*.
2. An `OPENAI_API_KEY` environment variable.
3. Otherwise the cell asks you to **paste the key** (hidden).

`get_secret()` is reused later for optional keys (Duffel, Microsoft Graph, Zoom, Slack, ngrok) -- if a key is missing, those tools run in **DRY RUN** mode.

In [2]:
import os, re, json, sqlite3, threading, time, warnings, logging
from datetime import datetime, timedelta
import requests
import pandas as pd
warnings.filterwarnings("ignore")
logging.disable(logging.CRITICAL)
os.environ["CREWAI_DISABLE_TELEMETRY"] = "true"
os.environ["OTEL_SDK_DISABLED"] = "true"

from crewai import Agent, Task, Crew, Process, LLM
from crewai.tools import tool

def get_secret(name, ask=False):
    """Read a secret from env vars or Colab Secrets. Optionally ask for it."""
    value = os.environ.get(name, "")
    if not value:
        try:
            from google.colab import userdata
            value = userdata.get(name) or ""
        except Exception:
            pass
    if not value and ask:
        from getpass import getpass
        value = getpass(f"Paste your {name}: ")
    return value.strip()

os.environ["OPENAI_API_KEY"] = get_secret("OPENAI_API_KEY", ask=True)
if not os.environ["OPENAI_API_KEY"].startswith("sk-"):
    raise ValueError("That does not look like an OpenAI API key -- it should start with 'sk-'.")

llm = LLM(model="gpt-4o-mini", temperature=0)

try:
    print("LLM check:", str(llm.call("Reply with exactly these words: KEY OK")).strip())
    print("Setup complete!")
except Exception as e:
    raise RuntimeError(f"OpenAI key check failed -- check the key, billing and internet. Details: {e}") from None

def run_agent(agent, description, expected_output):
    """Create a one-task crew, run it, and print the final answer."""
    task = Task(description=description, expected_output=expected_output, agent=agent)
    crew = Crew(agents=[agent], tasks=[task], process=Process.sequential, verbose=True)
    result = crew.kickoff()
    print("\n" + "=" * 70)
    print("FINAL ANSWER:")
    print(result)
    return result

Paste your OPENAI_API_KEY: ··········
LLM check: KEY OK
Setup complete!


---
# PART A -- Your First API: Build It, Test It in Postman, Give It to an Agent

The **Fee API** has just two endpoints:

| Method | Endpoint | Input | Output |
|---|---|---|---|
| GET | `/fee` | `?course=agentic ai` in the URL | `{"course": "agentic ai", "fee": 74999}` |
| POST | `/emi` | JSON body `{"fee": 74999, "months": 6}` | `{"monthly_emi": 12500, ...}` |

```
Step 3   Build the API          (Flask)
Step 4   Run it and test it     (inside Colab, with requests)
Step 5   Test it from Postman   (Colab → ngrok → Postman)
Step 6   Wrap it as tools       (@tool + requests)
Step 7   Give the tools to an agent
```

---
## Step 3 -- Build the Fee API (Flask)

- `Flask(...)` creates the API.
- `@fee_api.get("/fee")` means: *when someone sends GET to `/fee`, run this function.*
- `request.args` reads values from the **URL**; `request.get_json()` reads the **JSON body**.
- `jsonify(...)` sends JSON back. The number after it (`404`, `400`) is the **status code** -- default is `200 OK`.

In [4]:
from flask import Flask, request, jsonify

fee_api = Flask("fee_api")

FEES = {"python": 9999, "data analytics": 24999, "agentic ai": 74999}

@fee_api.get("/fee")
def get_fee():
    course = request.args.get("course", "").lower()
    match = next((name for name in FEES if name in course), None)
    if match is None:
        return jsonify({"error": f"Course '{course}' not found", "available": list(FEES)}), 404
    return jsonify({"course": match, "fee": FEES[match]})

@fee_api.post("/emi")
def calculate_emi():
    data = request.get_json(silent=True) or {}
    if "fee" not in data or "months" not in data:
        return jsonify({"error": "Send JSON with 'fee' and 'months'"}), 400
    return jsonify({"fee": data["fee"], "months": data["months"],
                    "monthly_emi": round(data["fee"] / data["months"])})

print("Fee API built -- 2 endpoints: GET /fee and POST /emi")

Fee API built -- 2 endpoints: GET /fee and POST /emi


---
## Step 4 -- Run the API and Test It Inside Colab

A Flask API normally runs in its own terminal. In a notebook we start it in a **background thread** on port **5001**, so the notebook stays usable.

Then we call it with `requests` -- the same thing Postman does.

In [5]:
FEE_API_URL = "http://127.0.0.1:5001"

try:
    requests.get(FEE_API_URL, timeout=1)                       # already running? then skip
except requests.ConnectionError:
    threading.Thread(target=fee_api.run, kwargs={"port": 5001}, daemon=True).start()
    time.sleep(1)

r = requests.get(f"{FEE_API_URL}/fee", params={"course": "agentic ai"})
print("GET  /fee?course=agentic ai  ->", r.status_code, r.json())

r = requests.get(f"{FEE_API_URL}/fee", params={"course": "cooking"})
print("GET  /fee?course=cooking     ->", r.status_code, r.json())

r = requests.post(f"{FEE_API_URL}/emi", json={"fee": 74999, "months": 6})
print("POST /emi                    ->", r.status_code, r.json())

 * Serving Flask app 'fee_api'
 * Debug mode: off
GET  /fee?course=agentic ai  -> 200 {'course': 'agentic ai', 'fee': 74999}
GET  /fee?course=cooking     -> 404 {'available': ['python', 'data analytics', 'agentic ai'], 'error': "Course 'cooking' not found"}
POST /emi                    -> 200 {'fee': 74999, 'monthly_emi': 12500, 'months': 6}


---
## Step 5 -- Test the API from Postman (Colab → ngrok → Postman)

**The problem:** the API runs on **Google's Colab machine**. `127.0.0.1` there is **not** your laptop, so Postman cannot reach it.

**The fix -- ngrok:** it creates a temporary **public HTTPS URL** that forwards to the API inside Colab.

```
Postman (your laptop / browser)  →  https://xxxx.ngrok-free.app  →  ngrok  →  Fee API on Colab (port 5001)
```

**One-time setup (2 minutes):**
1. Sign up free at **ngrok.com** → copy your **authtoken** from the dashboard
2. In Colab: key icon → add secret **`NGROK_AUTHTOKEN`** → turn on *Notebook access*
3. Run the cell below → copy the printed URL

**In Postman** (desktop app, or the web version at postman.com):

| # | Method | URL | Body | Expected |
|---|---|---|---|---|
| 1 | GET | `<your-ngrok-url>/fee?course=agentic ai` | -- | `200` + fee 74999 |
| 2 | GET | `<your-ngrok-url>/fee?course=cooking` | -- | `404` + error |
| 3 | POST | `<your-ngrok-url>/emi` | Body → **raw** → **JSON**: `{"fee": 74999, "months": 6}` | `200` + monthly_emi 12500 |

> If Postman shows an ngrok HTML warning page instead of JSON, add the header `ngrok-skip-browser-warning: true`.
> The URL is public while the tunnel is open -- use it only for demo data.

In [6]:
NGROK_AUTHTOKEN = get_secret("NGROK_AUTHTOKEN")

if NGROK_AUTHTOKEN:
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_AUTHTOKEN)
    fee_public_url = ngrok.connect(5001).public_url
    print("Use this base URL in Postman:", fee_public_url)
    print("Example:", f"{fee_public_url}/fee?course=agentic ai")
else:
    print("No NGROK_AUTHTOKEN found -- add it to Colab Secrets (see above) and re-run this cell.")

No NGROK_AUTHTOKEN found -- add it to Colab Secrets (see above) and re-run this cell.


---
## Step 6 -- Wrap the API as Tools

A tool is just a Python function that **calls the API** and returns the answer as text.

- The **docstring** tells the agent what the tool does and what input to send.
- The tool uses the **local** URL (`127.0.0.1:5001`) -- the agent runs in the same Colab as the API.
  Postman used the **ngrok** URL only because it runs on *your* laptop. Same API, two ways in.

Always test the tools directly before giving them to an agent.

In [7]:
@tool("Get Course Fee")
def get_course_fee(course: str) -> str:
    """Get the fee of a DV Analytics course from the Fee API.
    Input: course name. Example: agentic ai"""
    r = requests.get(f"{FEE_API_URL}/fee", params={"course": course}, timeout=10)
    return r.text

@tool("Calculate EMI")
def calculate_emi_tool(query: str) -> str:
    """Calculate the monthly EMI using the Fee API.
    Input format: fee,months. Example: 74999,6"""
    fee, months = [x.strip() for x in query.split(",")]
    r = requests.post(f"{FEE_API_URL}/emi", json={"fee": float(fee), "months": int(months)}, timeout=10)
    return r.text

print(get_course_fee.run("agentic ai"))
print(calculate_emi_tool.run("74999,6"))

{"course":"agentic ai","fee":74999}

{"fee":74999.0,"monthly_emi":12500,"months":6}



---
## Step 7 -- Give the Tools to an Agent

`tools=[get_course_fee, calculate_emi_tool]` gives the agent access to the API.

Now run the real-time example from the top of the notebook. In the verbose log, watch the agent:
1. call **Get Course Fee** → the tool sends `GET /fee` → gets 74999
2. call **Calculate EMI** with `74999,6` → the tool sends `POST /emi` → gets 12500
3. write the reply to the student

In [8]:
fee_agent = Agent(
    role="DV Analytics Fee Assistant",
    goal="Answer fee and EMI questions using the Fee API.",
    backstory="You never guess fees. You always use your tools to get the real numbers.",
    tools=[get_course_fee, calculate_emi_tool],
    llm=llm,
    verbose=True,
)

run_agent(
    fee_agent,
    description=("A student asks on WhatsApp: 'What is the fee for the Agentic AI course, "
                 "and how much per month if I pay in 6 EMIs?'"),
    expected_output="The course fee and the monthly EMI, in a short friendly reply.",
)

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.14.3                                                                                        │
│  Latest version:  1.15.27                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 50cb0a1a-f495-4f16-b9b6-f78dade41bed                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: A student asks on WhatsApp: 'What is the fee for the Agentic AI course, and how much per month if I pay  │
│  in 6 EMIs?'                                                                                                    │
│  ID: c9ce1ded-a86b-462c-befc-dd3078eb9f2f                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: DV Analytics Fee Assistant                                                                              │
│                                                                                                                 │
│  Task: A student asks on WhatsApp: 'What is the fee for the Agentic AI course, and how much per month if I pay  │
│  in 6 EMIs?'                                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: calculate_emi                                                                                            │
│  Args: {'query': '74999,6'}                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: DV Analytics Fee Assistant                                                                              │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The fee for the Agentic AI course is ₹74,999, and if you pay in 6 EMIs, it will be ₹12,500 per month.          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: A student asks on WhatsApp: 'What is the fee for the Agentic AI course, and how much per month if I pay  │
│  in 6 EMIs?'                                                                                                    │
│  Agent: DV Analytics Fee Assistant                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 50cb0a1a-f495-4f16-b9b6-f78dade41bed                                                                       │
│  Final Output: The fee for the Agentic AI course is ₹74,999, and if you pay in 6 EMIs, it will be ₹12,500 per   │
│  month.                                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


FINAL ANSWER:
The fee for the Agentic AI course is ₹74,999, and if you pay in 6 EMIs, it will be ₹12,500 per month.


CrewOutput(raw='The fee for the Agentic AI course is ₹74,999, and if you pay in 6 EMIs, it will be ₹12,500 per month.', pydantic=None, json_dict=None, tasks_output=[TaskOutput(description="A student asks on WhatsApp: 'What is the fee for the Agentic AI course, and how much per month if I pay in 6 EMIs?'", name="A student asks on WhatsApp: 'What is the fee for the Agentic AI course, and how much per month if I pay in 6 EMIs?'", expected_output='The course fee and the monthly EMI, in a short friendly reply.', summary="A student asks on WhatsApp: 'What is the fee for...", raw='The fee for the Agentic AI course is ₹74,999, and if you pay in 6 EMIs, it will be ₹12,500 per month.', pydantic=None, json_dict=None, agent='DV Analytics Fee Assistant', output_format=<OutputFormat.RAW: 'raw'>, messages=[{'role': 'system', 'content': 'You are DV Analytics Fee Assistant. You never guess fees. You always use your tools to get the real numbers.\nYour personal goal is: Answer fee and EMI questions usin

### What Just Happened

| Diagram step | What ran |
|---|---|
| Agent decides | GPT chose **Get Course Fee** with input `agentic ai` |
| Tool → API | `requests.get("http://127.0.0.1:5001/fee", params={"course": "agentic ai"})` |
| API → Tool | Flask returned `{"course": "agentic ai", "fee": 74999}` |
| Agent decides again | GPT chose **Calculate EMI** with input `74999,6` |
| Tool → API | `requests.post(".../emi", json={"fee": 74999, "months": 6})` → `{"monthly_emi": 12500}` |
| Agent replies | Fee and EMI in plain English |

**That is the whole pattern.** Everything else in this notebook -- the database API, Outlook, Zoom, SQL -- is this same loop with a different API behind the tool.

---
# PART B -- A Real API with a Database, an API Key and More Endpoints

Part A showed the idea with a tiny API. Real company APIs also have a **database**, an **API key** and **several endpoints**. Same steps, one level up -- **you play both roles** again:

```
Step 8   Create the company database        (the data the API serves)
Step 9   Build the API with Flask           (you are the backend developer)
Step 10  Run it and test it with requests   (same as Postman)
Step 11  Test it from Postman
Step 12  Give it to an agent as a tool      (you are the AI engineer)
```

---
## Step 8 -- Create the DV Analytics Database (the Data Behind the API)

Every API sits on top of data. We use **SQLite** -- a full SQL database stored in one file (`dv_analytics.db`), built into Python.

- In **Part A**, the Flask API reads courses from this database and saves new leads into it.
- In **Part C**, an agent will query the **same database directly with SQL**.

| Table | Contains |
|---|---|
| `courses` | 6 programs with category, duration and fee |
| `students` | 20 students with city and background |
| `enrollments` | 30 enrollments -- which student joined which course, batch, mode, status |
| `payments` | Payments made against each enrollment (full, partial, EMI) |

In [9]:
DB_PATH = "dv_analytics.db"
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)                        # start fresh every run

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

cur.executescript("""
CREATE TABLE courses (
    course_id      INTEGER PRIMARY KEY,
    name           TEXT NOT NULL,
    category       TEXT,
    duration_weeks INTEGER,
    fee            INTEGER                    -- in rupees
);
CREATE TABLE students (
    student_id  INTEGER PRIMARY KEY,
    name        TEXT NOT NULL,
    city        TEXT,
    email       TEXT,
    background  TEXT,                         -- Working professional / Fresher / College student
    joined_on   DATE
);
CREATE TABLE enrollments (
    enrollment_id INTEGER PRIMARY KEY,
    student_id    INTEGER REFERENCES students(student_id),
    course_id     INTEGER REFERENCES courses(course_id),
    batch_start   DATE,
    mode          TEXT,                       -- online / classroom
    status        TEXT                        -- active / completed / dropped
);
CREATE TABLE payments (
    payment_id    INTEGER PRIMARY KEY,
    enrollment_id INTEGER REFERENCES enrollments(enrollment_id),
    amount        INTEGER,                    -- in rupees
    paid_on       DATE,
    method        TEXT                        -- UPI / Card / EMI / NetBanking
);
""")

courses = [
    (1, "Python for Data Science",            "Programming",  6,  9999),
    (2, "Data Analytics",                     "Analytics",    8, 24999),
    (3, "Data Science and Machine Learning",  "Data Science", 16, 59999),
    (4, "Deep Learning and Generative AI",    "AI",          10, 49999),
    (5, "Agentic AI Professional",            "AI",          12, 74999),
    (6, "MLOps and Model Deployment",         "Engineering",  8, 39999),
]
students = [
    (1, "Priya Sharma", "Bengaluru", "Working professional"), (2, "Rahul Verma", "Pune", "Fresher"),
    (3, "Ananya Iyer", "Chennai", "Working professional"),    (4, "Karthik Reddy", "Hyderabad", "Working professional"),
    (5, "Sneha Patil", "Pune", "College student"),             (6, "Arjun Nair", "Bengaluru", "Working professional"),
    (7, "Meera Joshi", "Mumbai", "Working professional"),      (8, "Vikram Singh", "Delhi", "Working professional"),
    (9, "Divya Menon", "Bengaluru", "College student"),        (10, "Rohan Gupta", "Delhi", "Fresher"),
    (11, "Kavya Rao", "Hyderabad", "Working professional"),    (12, "Aditya Kulkarni", "Pune", "Working professional"),
    (13, "Nisha Agarwal", "Mumbai", "College student"),        (14, "Suresh Babu", "Chennai", "Working professional"),
    (15, "Pooja Hegde", "Bengaluru", "Working professional"),  (16, "Manoj Kumar", "Hyderabad", "Fresher"),
    (17, "Ritu Saxena", "Delhi", "Working professional"),      (18, "Ganesh Shetty", "Mumbai", "Working professional"),
    (19, "Lakshmi Prasad", "Chennai", "College student"),      (20, "Farhan Sheikh", "Bengaluru", "Working professional"),
]
students = [(sid, name, city, name.lower().replace(" ", ".") + "@example.com", bg, "2025-01-15")
            for sid, name, city, bg in students]

enrollments = [
    (1, 1, 2, "2025-03-10", "online", "completed"),   (2, 1, 5, "2026-08-03", "online", "active"),
    (3, 2, 1, "2026-07-06", "online", "completed"),   (4, 2, 2, "2026-09-07", "classroom", "active"),
    (5, 3, 3, "2026-06-01", "online", "active"),      (6, 4, 5, "2026-08-03", "online", "active"),
    (7, 5, 1, "2026-07-06", "online", "active"),      (8, 6, 5, "2026-09-05", "classroom", "active"),
    (9, 6, 6, "2026-10-12", "online", "active"),      (10, 7, 2, "2026-05-04", "online", "completed"),
    (11, 8, 3, "2026-06-01", "online", "dropped"),    (12, 9, 1, "2026-07-06", "classroom", "active"),
    (13, 10, 2, "2026-09-07", "online", "active"),    (14, 11, 5, "2026-08-03", "online", "active"),
    (15, 12, 4, "2026-07-13", "online", "active"),    (16, 13, 2, "2026-09-07", "classroom", "active"),
    (17, 14, 3, "2026-06-01", "online", "active"),    (18, 15, 5, "2026-09-05", "online", "active"),
    (19, 16, 1, "2026-07-06", "online", "dropped"),   (20, 17, 4, "2026-07-13", "online", "active"),
    (21, 18, 5, "2026-08-03", "classroom", "active"), (22, 19, 2, "2026-09-07", "online", "active"),
    (23, 20, 5, "2026-09-05", "online", "active"),    (24, 3, 5, "2026-09-05", "online", "active"),
    (25, 7, 3, "2026-08-10", "online", "active"),     (26, 11, 6, "2026-10-12", "online", "active"),
    (27, 14, 4, "2026-09-14", "online", "active"),    (28, 15, 3, "2026-04-06", "online", "completed"),
    (29, 17, 2, "2026-03-09", "classroom", "completed"), (30, 20, 3, "2026-04-06", "online", "dropped"),
]
# (enrollment_id, amount, paid_on, method) -- some full, some partial, some EMI, one unpaid (enrollment 24)
payment_rows = [
    (1, 24999, "2025-03-01", "UPI"),   (2, 74999, "2026-07-25", "UPI"),   (3, 9999, "2026-07-01", "Card"),
    (4, 12000, "2026-09-01", "UPI"),   (5, 59999, "2026-05-25", "NetBanking"),
    (6, 12500, "2026-08-01", "EMI"),   (6, 12500, "2026-09-01", "EMI"),   (6, 12500, "2026-10-01", "EMI"),
    (7, 9999, "2026-07-01", "UPI"),    (8, 74999, "2026-09-01", "Card"),  (9, 20000, "2026-10-05", "UPI"),
    (10, 24999, "2026-04-28", "Card"), (11, 15000, "2026-05-28", "UPI"),  (12, 9999, "2026-07-02", "UPI"),
    (13, 24999, "2026-09-01", "NetBanking"), (14, 25000, "2026-08-01", "UPI"), (15, 49999, "2026-07-10", "Card"),
    (16, 24999, "2026-09-02", "UPI"),  (17, 30000, "2026-05-30", "UPI"),  (18, 74999, "2026-09-01", "NetBanking"),
    (19, 5000, "2026-07-01", "UPI"),   (20, 49999, "2026-07-08", "Card"),
    (21, 12500, "2026-08-01", "EMI"),  (21, 12500, "2026-09-01", "EMI"),
    (22, 24999, "2026-09-03", "UPI"),  (23, 74999, "2026-09-02", "Card"), (25, 59999, "2026-08-05", "Card"),
    (26, 39999, "2026-10-08", "UPI"),  (27, 25000, "2026-09-10", "UPI"),  (28, 59999, "2026-04-01", "NetBanking"),
    (29, 24999, "2026-03-01", "UPI"),  (30, 20000, "2026-04-02", "UPI"),
]
payments = [(i + 1, *row) for i, row in enumerate(payment_rows)]

cur.executemany("INSERT INTO courses VALUES (?,?,?,?,?)", courses)
cur.executemany("INSERT INTO students VALUES (?,?,?,?,?,?)", students)
cur.executemany("INSERT INTO enrollments VALUES (?,?,?,?,?,?)", enrollments)
cur.executemany("INSERT INTO payments VALUES (?,?,?,?,?)", payments)
conn.commit()

for table in ["courses", "students", "enrollments", "payments"]:
    count = cur.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"{table:<12} {count} rows")
conn.close()

print("\nSample -- courses table:")
print(pd.read_sql("SELECT * FROM courses", sqlite3.connect(DB_PATH)).to_string(index=False))

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

courses      6 rows
students     20 rows
enrollments  30 rows
payments     32 rows

Sample -- courses table:
 course_id                              name     category  duration_weeks   fee
         1           Python for Data Science  Programming               6  9999
         2                    Data Analytics    Analytics               8 24999
         3 Data Science and Machine Learning Data Science              16 59999
         4   Deep Learning and Generative AI           AI              10 49999
         5           Agentic AI Professional           AI              12 74999
         6        MLOps and Model Deployment  Engineering               8 39999


---
## Step 9 -- Build the DV Analytics Admissions API (Flask)

**Flask** is a lightweight Python web framework. Each `@app.route` is one API **endpoint**.

| Method | Endpoint | What It Does | Auth |
|---|---|---|---|
| GET | `/api/health` | Is the API running? | No |
| GET | `/api/courses` | List courses (optional `?category=AI`) | No |
| GET | `/api/courses/<id>` | One course, or 404 | No |
| POST | `/api/leads` | Register a new enquiry (lead) | **API key** |
| GET | `/api/leads` | List all leads | **API key** |

**API key:** protected endpoints need the header `X-API-Key: dv-demo-key-123`. Without it → **401 Unauthorized**.

**Status codes** we return: `200 OK`, `201 Created`, `400 Bad Request`, `401 Unauthorized`, `404 Not Found`.

In [10]:
from flask import Flask, request, jsonify

API_KEY = "dv-demo-key-123"          # in production: from a secret store, never hard-coded
app = Flask("dv_admissions_api")

def db():
    con = sqlite3.connect(DB_PATH)
    con.row_factory = sqlite3.Row
    return con

with db() as con:                     # the API owns a new 'leads' table
    con.execute("""CREATE TABLE IF NOT EXISTS leads (
        lead_id INTEGER PRIMARY KEY AUTOINCREMENT, name TEXT, phone TEXT, city TEXT,
        course_id INTEGER, source TEXT, created_at TEXT)""")

def require_api_key():
    if request.headers.get("X-API-Key") != API_KEY:
        return jsonify({"error": "Unauthorized -- send a valid X-API-Key header"}), 401
    return None

@app.get("/api/health")
def health():
    return jsonify({"status": "ok", "service": "DV Analytics Admissions API"})

@app.get("/api/courses")
def list_courses():
    category = request.args.get("category")
    with db() as con:
        if category:
            rows = con.execute("SELECT * FROM courses WHERE category = ?", (category,)).fetchall()
        else:
            rows = con.execute("SELECT * FROM courses").fetchall()
    return jsonify([dict(r) for r in rows])

@app.get("/api/courses/<int:course_id>")
def get_course(course_id):
    with db() as con:
        row = con.execute("SELECT * FROM courses WHERE course_id = ?", (course_id,)).fetchone()
    if not row:
        return jsonify({"error": f"Course {course_id} not found"}), 404
    return jsonify(dict(row))

@app.post("/api/leads")
def create_lead():
    denied = require_api_key()
    if denied:
        return denied
    data = request.get_json(silent=True) or {}
    missing = [f for f in ("name", "phone", "city", "course_id") if not data.get(f)]
    if missing:
        return jsonify({"error": f"Missing fields: {missing}"}), 400
    with db() as con:
        if not con.execute("SELECT 1 FROM courses WHERE course_id = ?", (data["course_id"],)).fetchone():
            return jsonify({"error": f"course_id {data['course_id']} does not exist"}), 400
        cur = con.execute(
            "INSERT INTO leads (name, phone, city, course_id, source, created_at) VALUES (?,?,?,?,?,?)",
            (data["name"], data["phone"], data["city"], int(data["course_id"]),
             data.get("source", "api"), datetime.now().isoformat(timespec="seconds")))
        lead_id = cur.lastrowid
    return jsonify({"message": "Lead created", "lead_id": lead_id}), 201

@app.get("/api/leads")
def list_leads():
    denied = require_api_key()
    if denied:
        return denied
    with db() as con:
        rows = con.execute("SELECT * FROM leads ORDER BY lead_id").fetchall()
    return jsonify([dict(r) for r in rows])

print("API built. Endpoints:")
for rule in app.url_map.iter_rules():
    if rule.endpoint != "static":
        print(f"  {','.join(sorted(rule.methods - {'HEAD', 'OPTIONS'})):<5} {rule.rule}")

API built. Endpoints:
  GET   /api/health
  GET   /api/courses
  GET   /api/courses/<int:course_id>
  POST  /api/leads
  GET   /api/leads


---
## Step 10 -- Run the API and Test It with `requests`

In a normal project you run `python app.py` in a terminal. In a notebook, we run Flask in a **background thread** so the notebook stays usable.

Then we test every endpoint with the `requests` library. **This is exactly what Postman does** -- send an HTTP request, read the status code and JSON response.

| Postman | Python `requests` |
|---|---|
| Choose method (GET/POST) | `requests.get(...)` / `requests.post(...)` |
| Enter URL | first argument |
| Headers tab | `headers={...}` |
| Body → raw → JSON | `json={...}` |
| Status + response body | `r.status_code`, `r.json()` |

In [11]:
BASE_URL = "http://127.0.0.1:5000"

def api_is_up():
    try:
        return requests.get(f"{BASE_URL}/api/health", timeout=2).status_code == 200
    except requests.RequestException:
        return False

if not api_is_up():                       # start once; re-running this cell is safe
    threading.Thread(target=app.run,
                     kwargs=dict(host="0.0.0.0", port=5000, debug=False, use_reloader=False),
                     daemon=True).start()
    for _ in range(20):
        if api_is_up():
            break
        time.sleep(0.25)
print("API running at", BASE_URL, "->", api_is_up())

def show(label, r):
    print(f"{label:<44} {r.status_code}  {json.dumps(r.json())[:110]}")

HEADERS = {"X-API-Key": API_KEY}
print("\nTEST                                         STATUS RESPONSE")
print("-" * 100)
show("GET  /api/health",                      requests.get(f"{BASE_URL}/api/health"))
show("GET  /api/courses?category=AI",         requests.get(f"{BASE_URL}/api/courses", params={"category": "AI"}))
show("GET  /api/courses/5",                   requests.get(f"{BASE_URL}/api/courses/5"))
show("GET  /api/courses/99   (not found)",    requests.get(f"{BASE_URL}/api/courses/99"))
show("POST /api/leads        (no API key)",   requests.post(f"{BASE_URL}/api/leads", json={"name": "Test"}))
show("POST /api/leads        (missing fields)",
     requests.post(f"{BASE_URL}/api/leads", json={"name": "Test"}, headers=HEADERS))
show("POST /api/leads        (valid)",
     requests.post(f"{BASE_URL}/api/leads", headers=HEADERS,
                   json={"name": "Sneha Kulkarni", "phone": "+91 99000 11122", "city": "Pune",
                         "course_id": 2, "source": "postman-test"}))
show("GET  /api/leads",                       requests.get(f"{BASE_URL}/api/leads", headers=HEADERS))

 * Serving Flask app 'dv_admissions_api'
 * Debug mode: off
API running at http://127.0.0.1:5000 -> True

TEST                                         STATUS RESPONSE
----------------------------------------------------------------------------------------------------
GET  /api/health                             200  {"service": "DV Analytics Admissions API", "status": "ok"}
GET  /api/courses?category=AI                200  [{"category": "AI", "course_id": 4, "duration_weeks": 10, "fee": 49999, "name": "Deep Learning and Generative 
GET  /api/courses/5                          200  {"category": "AI", "course_id": 5, "duration_weeks": 12, "fee": 74999, "name": "Agentic AI Professional"}
GET  /api/courses/99   (not found)           404  {"error": "Course 99 not found"}
POST /api/leads        (no API key)          401  {"error": "Unauthorized -- send a valid X-API-Key header"}
POST /api/leads        (missing fields)      400  {"error": "Missing fields: ['phone', 'city', 'course_id']"}
POST

---
## Step 11 -- Test from Postman

**Where is the API running?** That decides how Postman reaches it.

### Option 1 -- Jupyter / VS Code on your laptop
The API runs on **your machine**. In Postman use `http://127.0.0.1:5000` directly.

### Option 2 -- Google Colab
The API runs on a **Google server**. Postman on your laptop **cannot** reach Colab's `127.0.0.1`.
Use **ngrok** exactly as in **Step 5** -- the next cell opens a public URL for port 5000.

> ngrok makes the API **public** while the tunnel is open. Only demo data here -- never expose real company data this way.

### Requests to Send in Postman

| # | Method | URL | Headers | Body (raw → JSON) | Expected |
|---|---|---|---|---|---|
| 1 | GET | `{{base_url}}/api/health` | -- | -- | 200 |
| 2 | GET | `{{base_url}}/api/courses?category=AI` | -- | -- | 200 |
| 3 | POST | `{{base_url}}/api/leads` | -- | `{"name":"Test"}` | **401** |
| 4 | POST | `{{base_url}}/api/leads` | `X-API-Key: dv-demo-key-123` | `{"name":"Asha Rao","phone":"+91 98450 22334","city":"Mysuru","course_id":5}` | **201** |
| 5 | GET | `{{base_url}}/api/leads` | `X-API-Key: dv-demo-key-123` | -- | 200 |

The cell after next also **writes a ready Postman collection** -- download it from the Colab Files panel and use **Postman → Import**.

In [12]:
# Postman from Colab -- open a public URL for the Admissions API (port 5000)
admissions_public_url = None

if NGROK_AUTHTOKEN:                                   # token was set in Part A
    for t in ngrok.get_tunnels():                     # free ngrok: keep one tunnel at a time
        ngrok.disconnect(t.public_url)
    admissions_public_url = ngrok.connect(5000).public_url
    print("Use this base URL in Postman:", admissions_public_url)
else:
    print("No NGROK_AUTHTOKEN -- see Part A, Step 5. On your laptop, use http://127.0.0.1:5000 in Postman.")

No NGROK_AUTHTOKEN -- see Part A, Step 5. On your laptop, use http://127.0.0.1:5000 in Postman.


In [13]:
# Write a Postman collection you can import (Postman -> Import -> select this file)
def pm_request(name, method, path, body=None, auth=False):
    req = {"method": method,
           "header": ([{"key": "X-API-Key", "value": "{{api_key}}"}] if auth else []) +
                     ([{"key": "Content-Type", "value": "application/json"}] if body else []),
           "url": {"raw": "{{base_url}}" + path, "host": ["{{base_url}}"],
                   "path": path.split("?")[0].strip("/").split("/"),
                   **({"query": [{"key": k, "value": v} for k, v in
                                 (p.split("=") for p in path.split("?")[1].split("&"))]} if "?" in path else {})}}
    if body:
        req["body"] = {"mode": "raw", "raw": json.dumps(body, indent=2), "options": {"raw": {"language": "json"}}}
    return {"name": name, "request": req}

collection = {
    "info": {"name": "DV Analytics Admissions API",
             "schema": "https://schema.getpostman.com/json/collection/v2.1.0/collection.json"},
    "variable": [{"key": "base_url", "value": admissions_public_url or BASE_URL},
                 {"key": "api_key", "value": API_KEY}],
    "item": [
        pm_request("1 Health check", "GET", "/api/health"),
        pm_request("2 List AI courses", "GET", "/api/courses?category=AI"),
        pm_request("3 Create lead WITHOUT key (expect 401)", "POST", "/api/leads", {"name": "Test"}),
        pm_request("4 Create lead (expect 201)", "POST", "/api/leads",
                   {"name": "Asha Rao", "phone": "+91 98450 22334", "city": "Mysuru", "course_id": 5}, auth=True),
        pm_request("5 List leads", "GET", "/api/leads", auth=True),
    ],
}
# Request 3 must NOT carry the key -- that is the point of the test
with open("DV_Analytics_API.postman_collection.json", "w") as f:
    json.dump(collection, f, indent=2)
print("Saved DV_Analytics_API.postman_collection.json")
print("base_url in the collection:", collection["variable"][0]["value"])

Saved DV_Analytics_API.postman_collection.json
base_url in the collection: http://127.0.0.1:5000


---
## Step 12 -- Give Your API to an Agent

Now switch roles: you are the **AI engineer**. The API is just another system the agent can use.

Two tools wrap the API with `requests`:

| Tool | Calls |
|---|---|
| `List Courses (DV API)` | `GET /api/courses` |
| `Register Lead (DV API)` | `POST /api/leads` with the API key |

**Good API-tool habits shown here:** a `timeout` on every request, the API key added **inside the tool** (the LLM never sees it), and clear error messages the agent can act on.

**Task:** a WhatsApp enquiry comes in. The Admissions Agent must find the right course **from the API** and **register the lead** through the API.

In [14]:
@tool("List Courses (DV API)")
def list_courses_api(category: str = "all") -> str:
    """List DV Analytics courses from the Admissions API with course_id, fee and duration.
    Input: a category (Programming, Analytics, Data Science, AI, Engineering) or 'all'."""
    params = {} if category.strip().lower() in ("", "all") else {"category": category.strip()}
    try:
        r = requests.get(f"{BASE_URL}/api/courses", params=params, timeout=10)
        r.raise_for_status()
        return json.dumps(r.json(), indent=1)
    except requests.RequestException as e:
        return f"Admissions API error: {e}"

@tool("Register Lead (DV API)")
def register_lead_api(details: str) -> str:
    """Register a new student enquiry (lead) in the Admissions API.
    Input format: name,phone,city,course_id
    Example: Rahul Verma,+91 98860 12345,Pune,2
    Returns: the new lead_id, or the error from the API."""
    parts = [p.strip() for p in details.split(",")]
    if len(parts) != 4:
        return "Input must be exactly: name,phone,city,course_id"
    payload = {"name": parts[0], "phone": parts[1], "city": parts[2],
               "course_id": parts[3], "source": "whatsapp-agent"}
    try:
        r = requests.post(f"{BASE_URL}/api/leads", json=payload,
                          headers={"X-API-Key": API_KEY}, timeout=10)   # key added here, not by the LLM
        return f"HTTP {r.status_code}: {r.json()}"
    except requests.RequestException as e:
        return f"Admissions API error: {e}"

print(list_courses_api.run("Analytics"))       # test directly first

admissions_agent = Agent(
    role="Admissions Counsellor at DV Analytics",
    goal="Recommend the right course using live course data and register every enquiry as a lead.",
    backstory=(
        "You handle WhatsApp enquiries. Always fetch courses from the Admissions API -- never invent "
        "fees or course IDs. After recommending a course, register the lead with the correct course_id."
    ),
    tools=[list_courses_api, register_lead_api],
    llm=llm,
    verbose=True,
)

run_agent(
    admissions_agent,
    description=(
        "WhatsApp enquiry from Rahul Verma, phone +91 98860 12345, from Pune: "
        "'I am a commerce graduate with no coding background. I want to become a data analyst. "
        "Which course should I join and what is the fee?' "
        "Recommend one course using the API, then register Rahul as a lead for that course."
    ),
    expected_output="Course recommendation with fee and duration, plus the lead ID from the API.",
)

print("\nLeads now stored by the API:")
for lead in requests.get(f"{BASE_URL}/api/leads", headers=HEADERS, timeout=10).json():
    print("  ", lead)

[
 {
  "category": "Analytics",
  "course_id": 2,
  "duration_weeks": 8,
  "fee": 24999,
  "name": "Data Analytics"
 }
]


╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.14.3                                                                                        │
│  Latest version:  1.15.27                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: d98a417a-d1c9-4714-89a7-fbea219fbb22                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: WhatsApp enquiry from Rahul Verma, phone +91 98860 12345, from Pune: 'I am a commerce graduate with no   │
│  coding background. I want to become a data analyst. Which course should I join and what is the fee?'           │
│  Recommend one course using the API, then register Rahul as a lead for that course.                             │
│  ID: 275328fc-f02a-418d-bad9-5a545a3f26cb                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Admissions Counsellor at DV Analytics                                                                   │
│                                                                                                                 │
│  Task: WhatsApp enquiry from Rahul Verma, phone +91 98860 12345, from Pune: 'I am a commerce graduate with no   │
│  coding background. I want to become a data analyst. Which course should I join and what is the fee?'           │
│  Recommend one course using the API, then register Rahul as a lead for that course.                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: WhatsApp enquiry from Rahul Verma, phone +91 98860 12345, from Pune: 'I am a commerce graduate with no   │
│  coding background. I want to become a data analyst. Which course should I join and what is the fee?'           │
│  Recommend one course using the API, then register Rahul as a lead for that course.                             │
│  Agent: Admissions Counsellor at DV Analytics                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: d98a417a-d1c9-4714-89a7-fbea219fbb22                                                                       │
│  Final Output: You should join the "Data Analytics" course, which has a duration of 8 weeks and a fee of        │
│  ₹24,999. Your lead ID is 2.                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


FINAL ANSWER:
You should join the "Data Analytics" course, which has a duration of 8 weeks and a fee of ₹24,999. Your lead ID is 2.

Leads now stored by the API:


   {'city': 'Pune', 'course_id': 2, 'created_at': '2026-10-13T17:14:40', 'lead_id': 1, 'name': 'Sneha Kulkarni', 'phone': '+91 99000 11122', 'source': 'postman-test'}
   {'city': 'Pune', 'course_id': 2, 'created_at': '2026-10-13T17:14:43', 'lead_id': 2, 'name': 'Rahul Verma', 'phone': '+91 98860 12345', 'source': 'whatsapp-agent'}


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

---
# PART C -- Real-Time External APIs as Tools (HTTP with `requests`)

Every SaaS product -- travel sites, Outlook, Zoom, Slack, payment gateways -- exposes an **HTTP API**.
In Parts A and B you called **your own** APIs. Here the APIs belong to **other companies** -- the pattern is the same, plus **authentication**. If you can make an HTTP request, you can turn it into an agent tool.

---
## Step 13 -- How an HTTP API Tool Works

Every API call has the same parts:

| Part | Example | In `requests` |
|---|---|---|
| **Method** | `GET` (read), `POST` (create), `PATCH` (update), `DELETE` | `requests.get / post / patch / delete` |
| **URL** | `https://api.zoom.us/v2/users/me/meetings` | first argument |
| **Query params** | `?city=Goa&days=3` | `params={...}` |
| **Headers** | `Authorization: Bearer <token>`, `Content-Type: application/json` | `headers={...}` |
| **Body** | `{"topic": "Fee counselling", "duration": 30}` | `json={...}` |
| **Status code** | `200 OK`, `201 Created`, `401`, `404`, `429 Too Many Requests`, `500` | `r.status_code` |
| **Response** | JSON data | `r.json()` |

**Authentication types you will meet:**

| Type | How | Used by |
|---|---|---|
| No auth | Just call the URL | Open-Meteo, exchange rates |
| API key | `X-API-Key: ...` header or `?key=...` | Your Flask API, many data APIs |
| Bearer token | `Authorization: Bearer <token>` | Duffel, Microsoft Graph |
| OAuth 2.0 | Exchange client ID + secret for a short-lived token | Zoom, Microsoft Graph (production), Google |
| Webhook URL | The secret is inside the URL | Slack incoming webhooks |

### DRY RUN Mode

Steps 15-18 need accounts and keys. So that **every cell runs for everyone**, each tool has a **DRY RUN** mode:
no key → the tool returns the **exact HTTP request** it would send (token masked). Add the key → the same tool makes the **live** call.

In [15]:
def mask(value):
    value = str(value)
    return value[:6] + "..." + value[-4:] if len(value) > 14 else "***"

def http_call(service, method, url, *, live, headers=None, params=None, body=None, timeout=15):
    """Make a live HTTP call -- or, with no credentials, return a DRY RUN preview of the same request."""
    headers = headers or {}
    if not live:
        shown = {k: (("Bearer " + mask(v[7:])) if str(v).startswith("Bearer ") else
                     (mask(v) if k.lower() in ("authorization", "x-api-key") else v))
                 for k, v in headers.items()}
        preview = {"method": method, "url": url, "headers": shown}
        if params:
            preview["params"] = params
        if body:
            preview["json_body"] = body
        return f"DRY RUN ({service} credentials not set) -- this exact request would be sent:\n" + \
               json.dumps(preview, indent=2)
    try:
        r = requests.request(method, url, headers=headers, params=params, json=body, timeout=timeout)
        if r.status_code == 401:
            return f"{service} error 401 Unauthorized -- the token is missing, wrong or expired."
        if r.status_code == 429:
            return f"{service} error 429 Too Many Requests -- wait and try again."
        r.raise_for_status()
        return r.json() if r.content else {"status": r.status_code}
    except requests.RequestException as e:
        return f"{service} request failed: {e}"

print(http_call("Demo", "POST", "https://api.example.com/v1/items", live=False,
                headers={"Authorization": "Bearer abcdef1234567890xyz"}, body={"name": "test"}))

DRY RUN (Demo credentials not set) -- this exact request would be sent:
{
  "method": "POST",
  "url": "https://api.example.com/v1/items",
  "headers": {
    "Authorization": "Bearer abcdef...0xyz"
  },
  "json_body": {
    "name": "test"
  }
}


---
## Step 14 -- Live Free APIs: Weather and Currency (No Key)

These tools call **real live APIs** -- no registration needed. Run in Colab where internet is available.

| API | Data | Auth |
|---|---|---|
| **Open-Meteo** | Geocoding (city → latitude/longitude) + 3-day weather forecast | None |
| **open.er-api.com** | Live currency exchange rates | None |

Notice the **two-call pattern** in the weather tool: first convert the city name to coordinates, then get the forecast. Tools often chain several API calls internally.

In [16]:
@tool("Weather Forecast")
def weather_forecast(city: str) -> str:
    """Get a live 3-day weather forecast for any city worldwide (Open-Meteo, no key needed).
    Input: city name. Example: Goa"""
    try:
        geo = requests.get("https://geocoding-api.open-meteo.com/v1/search",
                           params={"name": city.strip(), "count": 1}, timeout=10).json()
        if not geo.get("results"):
            return f"City '{city}' not found."
        place = geo["results"][0]
        fc = requests.get("https://api.open-meteo.com/v1/forecast", timeout=10, params={
            "latitude": place["latitude"], "longitude": place["longitude"], "timezone": "auto",
            "forecast_days": 3,
            "daily": "temperature_2m_max,temperature_2m_min,precipitation_probability_max"}).json()
        d = fc["daily"]
        lines = [f"{d['time'][i]}: {d['temperature_2m_min'][i]}-{d['temperature_2m_max'][i]} C, "
                 f"rain chance {d['precipitation_probability_max'][i]}%" for i in range(len(d["time"]))]
        return f"Live forecast for {place['name']}, {place.get('country', '')}:\n" + "\n".join(lines)
    except Exception as e:
        return f"Weather service unavailable right now ({type(e).__name__})."

@tool("Convert Currency")
def convert_currency(query: str) -> str:
    """Convert money between currencies using live exchange rates (no key needed).
    Input format: amount,FROM,TO. Example: 50000,INR,USD"""
    try:
        amount, src, dst = [p.strip() for p in query.split(",")]
        data = requests.get(f"https://open.er-api.com/v6/latest/{src.upper()}", timeout=10).json()
        rate = data["rates"][dst.upper()]
        return f"{amount} {src.upper()} = {round(float(amount) * rate, 2)} {dst.upper()} (live rate {rate})"
    except Exception as e:
        return f"Currency service unavailable or bad input ({type(e).__name__}). Use: amount,FROM,TO"

print(weather_forecast.run("Goa"))
print(convert_currency.run("50000,INR,USD"))

Live forecast for Genoa, Italy:
2026-10-13: 16.4-23.1 C, rain chance 0%
2026-10-14: 15.0-22.5 C, rain chance 0%
2026-10-15: 15.5-23.1 C, rain chance 0%
50000 INR = 521.35 USD (live rate 0.010427)


---
## Step 15 -- Flight Search API: Duffel (and Why Not MakeMyTrip)

**Can an agent call MakeMyTrip?** Not directly. MakeMyTrip does **not** offer a public developer API -- access is only through **business partner agreements**. The same is true for most big travel sites.

**What developers use instead:** flight-booking APIs built for developers. The popular free self-service option, **Amadeus Self-Service, was shut down in July 2026**. This notebook uses **Duffel**, which has a free **test mode** (sign up at duffel.com → copy a *test* access token → add `DUFFEL_TOKEN` to Colab Secrets).

| | |
|---|---|
| **Endpoint** | `POST https://api.duffel.com/air/offer_requests?return_offers=true` |
| **Auth** | `Authorization: Bearer <token>` + `Duffel-Version: v2` header |
| **Body** | slices (origin, destination, date) + passengers |
| **Test mode** | Returns realistic sample offers from "Duffel Airways" -- no real booking, no charges |

No token → **DRY RUN** shows the exact request. Check Duffel's docs if a field name has changed.

In [17]:
DUFFEL_TOKEN = get_secret("DUFFEL_TOKEN")

@tool("Search Flights (Duffel)")
def search_flights_duffel(query: str) -> str:
    """Search live flight offers through the Duffel API.
    Input format: origin_IATA,destination_IATA,YYYY-MM-DD
    Example: BOM,GOI,2026-11-20  (BOM = Mumbai, GOI = Goa, BLR = Bengaluru, DEL = Delhi)
    Returns: the cheapest offers with airline, departure time and price."""
    try:
        origin, destination, date = [p.strip().upper() for p in query.split(",")]
    except ValueError:
        return "Input must be: origin_IATA,destination_IATA,YYYY-MM-DD"
    result = http_call(
        "Duffel", "POST", "https://api.duffel.com/air/offer_requests",
        live=bool(DUFFEL_TOKEN),
        params={"return_offers": "true", "supplier_timeout": "10000"},
        headers={"Authorization": f"Bearer {DUFFEL_TOKEN or 'YOUR_DUFFEL_TOKEN'}",
                 "Duffel-Version": "v2", "Content-Type": "application/json", "Accept": "application/json"},
        body={"data": {"slices": [{"origin": origin, "destination": destination, "departure_date": date}],
                       "passengers": [{"type": "adult"}], "cabin_class": "economy"}})
    if not isinstance(result, dict):
        return result                                   # DRY RUN preview or error message
    offers = sorted(result.get("data", {}).get("offers", []), key=lambda o: float(o["total_amount"]))[:3]
    if not offers:
        return f"No offers found for {origin} -> {destination} on {date}."
    return "\n".join(f"{o['owner']['name']} | departs {o['slices'][0]['segments'][0]['departing_at']} | "
                     f"{o['total_amount']} {o['total_currency']}" for o in offers)

print(search_flights_duffel.run("BOM,GOI,2026-11-20"))

DRY RUN (Duffel credentials not set) -- this exact request would be sent:
{
  "method": "POST",
  "url": "https://api.duffel.com/air/offer_requests",
  "headers": {
    "Authorization": "Bearer YOUR_D...OKEN",
    "Duffel-Version": "v2",
    "Content-Type": "application/json",
    "Accept": "application/json"
  },
  "params": {
    "return_offers": "true",
    "supplier_timeout": "10000"
  },
  "json_body": {
    "data": {
      "slices": [
        {
          "origin": "BOM",
          "destination": "GOI",
          "departure_date": "2026-11-20"
        }
      ],
      "passengers": [
        {
          "type": "adult"
        }
      ],
      "cabin_class": "economy"
    }
  }
}


### Travel Agent with Live Tools

The same Travel Agent from Section 7 -- now with a **live** forecast, **live** currency and a **real** flight API.

In [18]:
travel_agent = Agent(
    role="Travel Planning Assistant",
    goal="Plan trips using live data -- flights, weather and currency.",
    backstory="You always fetch live data with your tools. Never invent prices or forecasts.",
    tools=[search_flights_duffel, weather_forecast, convert_currency],
    llm=llm,
    verbose=True,
)

run_agent(
    travel_agent,
    description=(
        "Plan a trip from Mumbai (BOM) to Goa (GOI) on 2026-11-20 for one adult. "
        "Search flights, check the Goa weather forecast, and convert a Rs 25,000 budget to USD "
        "for a friend joining from the US. Summarise clearly. If a tool is in DRY RUN mode, say so."
    ),
    expected_output="Flight options (or the DRY RUN note), Goa forecast, currency conversion, and a short trip summary.",
)

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.14.3                                                                                        │
│  Latest version:  1.15.27                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 07cc1824-b1ed-4f6b-ac13-ae9b7efa68c7                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Plan a trip from Mumbai (BOM) to Goa (GOI) on 2026-11-20 for one adult. Search flights, check the Goa    │
│  weather forecast, and convert a Rs 25,000 budget to USD for a friend joining from the US. Summarise clearly.   │
│  If a tool is in DRY RUN mode, say so.                                                                          │
│  ID: 09187632-42c8-43d9-9ae6-f10e610bc234                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Travel Planning Assistant                                                                               │
│                                                                                                                 │
│  Task: Plan a trip from Mumbai (BOM) to Goa (GOI) on 2026-11-20 for one adult. Search flights, check the Goa    │
│  weather forecast, and convert a Rs 25,000 budget to USD for a friend joining from the US. Summarise clearly.   │
│  If a tool is in DRY RUN mode, say so.                                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: search_flights_duffel                                                                                    │
│  Args: {'query': 'BOM,GOI,2026-11-20'}                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Plan a trip from Mumbai (BOM) to Goa (GOI) on 2026-11-20 for one adult. Search flights, check the Goa    │
│  weather forecast, and convert a Rs 25,000 budget to USD for a friend joining from the US. Summarise clearly.   │
│  If a tool is in DRY RUN mode, say so.                                                                          │
│  Agent: Travel Planning Assistant                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 07cc1824-b1ed-4f6b-ac13-ae9b7efa68c7                                                                       │
│  Final Output: **Flight Options:** DRY RUN (Duffel credentials not set) -- flight search could not be           │
│  completed.                                                                                                     │
│                                                                                                                 │
│  **Goa Weather Forecast:** Weather service unavailable right now (ConnectTimeout).                              │
│                                                                                                                 │
│  **Currency Conversion:** 25,000 INR = 260.68 USD (live rate 0.010427).                                         │
│                                                                                                                 │
│  **Trip Summary:**                                                                                              │
│  - Departure: Mumbai (BOM) to Goa (GOI) on 2026-11-20 for one adult.                                            │
│  - Flight options are currently unavailable due to a DRY RUN.                                                   │
│  - Weather forecast for Goa could not be retrieved.                                                             │
│  - Budget of Rs 25,000 converts to approximately $260.68 USD.                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


FINAL ANSWER:
**Flight Options:** DRY RUN (Duffel credentials not set) -- flight search could not be completed.

**Goa Weather Forecast:** Weather service unavailable right now (ConnectTimeout).

**Currency Conversion:** 25,000 INR = 260.68 USD (live rate 0.010427).

**Trip Summary:** 
- Departure: Mumbai (BOM) to Goa (GOI) on 2026-11-20 for one adult.
- Flight options are currently unavailable due to a DRY RUN.
- Weather forecast for Goa could not be retrieved.
- Budget of Rs 25,000 converts to approximately $260.68 USD.


CrewOutput(raw='**Flight Options:** DRY RUN (Duffel credentials not set) -- flight search could not be completed.\n\n**Goa Weather Forecast:** Weather service unavailable right now (ConnectTimeout).\n\n**Currency Conversion:** 25,000 INR = 260.68 USD (live rate 0.010427).\n\n**Trip Summary:** \n- Departure: Mumbai (BOM) to Goa (GOI) on 2026-11-20 for one adult.\n- Flight options are currently unavailable due to a DRY RUN.\n- Weather forecast for Goa could not be retrieved.\n- Budget of Rs 25,000 converts to approximately $260.68 USD.', pydantic=None, json_dict=None, tasks_output=[TaskOutput(description='Plan a trip from Mumbai (BOM) to Goa (GOI) on 2026-11-20 for one adult. Search flights, check the Goa weather forecast, and convert a Rs 25,000 budget to USD for a friend joining from the US. Summarise clearly. If a tool is in DRY RUN mode, say so.', name='Plan a trip from Mumbai (BOM) to Goa (GOI) on 2026-11-20 for one adult. Search flights, check the Goa weather forecast, and convert 

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

---
## Step 16 -- Outlook Tools: Microsoft Graph (Email + Calendar)

Outlook email and calendar are reached through **Microsoft Graph** -- one API for all of Microsoft 365.

| Tool | Method + Endpoint | Success |
|---|---|---|
| Send email | `POST https://graph.microsoft.com/v1.0/me/sendMail` | `202 Accepted` |
| Create calendar event | `POST https://graph.microsoft.com/v1.0/me/events` | `201 Created` |
| Read calendar | `GET https://graph.microsoft.com/v1.0/me/calendarView` | `200 OK` |

**Getting a token:**
- **For learning (5 minutes):** open **Microsoft Graph Explorer**, sign in, grant `Mail.Send` and `Calendars.ReadWrite`, copy the **access token** → add `MS_GRAPH_TOKEN` to Colab Secrets. It expires in about **1 hour**.
- **For production:** register an app in **Microsoft Entra ID (Azure AD)**, and use the **MSAL** library to get tokens automatically with the right permissions.

No token → **DRY RUN**.

In [19]:
MS_GRAPH_TOKEN = get_secret("MS_GRAPH_TOKEN")
GRAPH = "https://graph.microsoft.com/v1.0"
GRAPH_HEADERS = {"Authorization": f"Bearer {MS_GRAPH_TOKEN or 'YOUR_GRAPH_TOKEN'}",
                 "Content-Type": "application/json"}

@tool("Send Outlook Email")
def send_outlook_email(details: str) -> str:
    """Send an email from the signed-in Outlook account via Microsoft Graph.
    Input format: to_email|subject|body   (separated by the | character)
    Example: anita@dvanalytics.example|Fee follow-up|Please call these students today."""
    try:
        to, subject, body = [p.strip() for p in details.split("|", 2)]
    except ValueError:
        return "Input must be: to_email|subject|body"
    result = http_call("Microsoft Graph", "POST", f"{GRAPH}/me/sendMail", live=bool(MS_GRAPH_TOKEN),
                       headers=GRAPH_HEADERS,
                       body={"message": {"subject": subject,
                                         "body": {"contentType": "Text", "content": body},
                                         "toRecipients": [{"emailAddress": {"address": to}}]},
                             "saveToSentItems": True})
    return result if isinstance(result, str) else f"Email sent to {to}."

@tool("Create Outlook Event")
def create_outlook_event(details: str) -> str:
    """Create an Outlook calendar event and invite an attendee via Microsoft Graph.
    Input format: subject|start (YYYY-MM-DDTHH:MM, India time)|duration_minutes|attendee_email|description
    Example: Fee counselling|2026-10-14T11:00|30|anita@dvanalytics.example|Zoom link: https://..."""
    try:
        subject, start, minutes, attendee, description = [p.strip() for p in details.split("|", 4)]
        start_dt = datetime.fromisoformat(start)
        end_dt = start_dt + timedelta(minutes=int(minutes))
    except ValueError:
        return "Input must be: subject|YYYY-MM-DDTHH:MM|duration_minutes|attendee_email|description"
    result = http_call("Microsoft Graph", "POST", f"{GRAPH}/me/events", live=bool(MS_GRAPH_TOKEN),
                       headers=GRAPH_HEADERS,
                       body={"subject": subject,
                             "body": {"contentType": "Text", "content": description},
                             "start": {"dateTime": start_dt.isoformat(), "timeZone": "India Standard Time"},
                             "end": {"dateTime": end_dt.isoformat(), "timeZone": "India Standard Time"},
                             "attendees": [{"emailAddress": {"address": attendee}, "type": "required"}]})
    return result if isinstance(result, str) else f"Event created: {result.get('webLink', subject)}"

print(send_outlook_email.run("anita@dvanalytics.example|Test from agent tools|Hello from the notebook."))

DRY RUN (Microsoft Graph credentials not set) -- this exact request would be sent:
{
  "method": "POST",
  "url": "https://graph.microsoft.com/v1.0/me/sendMail",
  "headers": {
    "Authorization": "Bearer YOUR_G...OKEN",
    "Content-Type": "application/json"
  },
  "json_body": {
    "message": {
      "subject": "Test from agent tools",
      "body": {
        "contentType": "Text",
        "content": "Hello from the notebook."
      },
      "toRecipients": [
        {
          "emailAddress": {
            "address": "anita@dvanalytics.example"
          }
        }
      ]
    },
    "saveToSentItems": true
  }
}


---
## Step 17 -- Zoom Tool: Create a Meeting

Zoom uses **Server-to-Server OAuth** -- a two-step pattern you will see in many enterprise APIs:

```
Step 1: POST https://zoom.us/oauth/token?grant_type=account_credentials&account_id=<ACCOUNT_ID>
        Basic auth with CLIENT_ID : CLIENT_SECRET          →  short-lived access_token (1 hour)

Step 2: POST https://api.zoom.us/v2/users/me/meetings
        Authorization: Bearer <access_token>               →  201 Created + join_url
```

**Setup (free Zoom account):** Zoom App Marketplace → *Develop* → *Build App* → **Server-to-Server OAuth** → add scope for creating meetings → copy **Account ID, Client ID, Client Secret** → add `ZOOM_ACCOUNT_ID`, `ZOOM_CLIENT_ID`, `ZOOM_CLIENT_SECRET` to Colab Secrets.

No credentials → **DRY RUN**.

In [20]:
ZOOM_ACCOUNT_ID = get_secret("ZOOM_ACCOUNT_ID")
ZOOM_CLIENT_ID = get_secret("ZOOM_CLIENT_ID")
ZOOM_CLIENT_SECRET = get_secret("ZOOM_CLIENT_SECRET")
ZOOM_LIVE = all([ZOOM_ACCOUNT_ID, ZOOM_CLIENT_ID, ZOOM_CLIENT_SECRET])

def zoom_access_token():
    r = requests.post("https://zoom.us/oauth/token", timeout=15,
                      params={"grant_type": "account_credentials", "account_id": ZOOM_ACCOUNT_ID},
                      auth=(ZOOM_CLIENT_ID, ZOOM_CLIENT_SECRET))
    r.raise_for_status()
    return r.json()["access_token"]

@tool("Create Zoom Meeting")
def create_zoom_meeting(details: str) -> str:
    """Create a scheduled Zoom meeting and return its join link.
    Input format: topic|start (YYYY-MM-DDTHH:MM, India time)|duration_minutes
    Example: Fee counselling session|2026-10-14T11:00|30"""
    try:
        topic, start, minutes = [p.strip() for p in details.split("|")]
        datetime.fromisoformat(start)
    except ValueError:
        return "Input must be: topic|YYYY-MM-DDTHH:MM|duration_minutes"
    try:
        token = zoom_access_token() if ZOOM_LIVE else "YOUR_ZOOM_ACCESS_TOKEN"
    except requests.RequestException as e:
        return f"Zoom authentication failed: {e}"
    result = http_call("Zoom", "POST", "https://api.zoom.us/v2/users/me/meetings", live=ZOOM_LIVE,
                       headers={"Authorization": f"Bearer {token}", "Content-Type": "application/json"},
                       body={"topic": topic, "type": 2, "start_time": f"{start}:00", "duration": int(minutes),
                             "timezone": "Asia/Kolkata",
                             "settings": {"waiting_room": True, "join_before_host": False}})
    if isinstance(result, str):
        return result + "\n(DRY RUN placeholder join link: https://zoom.us/j/0000000000)" \
            if result.startswith("DRY RUN") else result
    return f"Zoom meeting created: {result['join_url']} (meeting ID {result['id']})"

print(create_zoom_meeting.run("Fee counselling session|2026-10-14T11:00|30"))

DRY RUN (Zoom credentials not set) -- this exact request would be sent:
{
  "method": "POST",
  "url": "https://api.zoom.us/v2/users/me/meetings",
  "headers": {
    "Authorization": "Bearer YOUR_Z...OKEN",
    "Content-Type": "application/json"
  },
  "json_body": {
    "topic": "Fee counselling session",
    "type": 2,
    "start_time": "2026-10-14T11:00:00",
    "duration": 30,
    "timezone": "Asia/Kolkata",
    "settings": {
      "waiting_room": true,
      "join_before_host": false
    }
  }
}
(DRY RUN placeholder join link: https://zoom.us/j/0000000000)


---
## Step 18 -- Slack Tool: Post a Team Update

The simplest real integration: a Slack **incoming webhook**.
The secret is the URL itself -- one `POST` with `{"text": "..."}` posts to a channel.

**Setup:** Slack → *Apps* → **Incoming Webhooks** → choose a channel → copy the webhook URL → add `SLACK_WEBHOOK_URL` to Colab Secrets.

The same pattern works for **Microsoft Teams** and **Google Chat** webhooks.

In [21]:
SLACK_WEBHOOK_URL = get_secret("SLACK_WEBHOOK_URL")

@tool("Post to Slack")
def post_to_slack(message: str) -> str:
    """Post a message to the team's Slack channel.
    Input: the message text."""
    result = http_call("Slack", "POST", SLACK_WEBHOOK_URL or "https://hooks.slack.com/services/YOUR/WEBHOOK/URL",
                       live=bool(SLACK_WEBHOOK_URL), body={"text": message})
    return result if isinstance(result, str) else "Posted to Slack."

print(post_to_slack.run("Test message from the DV Analytics agent notebook."))

DRY RUN (Slack credentials not set) -- this exact request would be sent:
{
  "method": "POST",
  "url": "https://hooks.slack.com/services/YOUR/WEBHOOK/URL",
  "headers": {},
  "json_body": {
    "text": "Test message from the DV Analytics agent notebook."
  }
}


---
# PART D -- Agents on a SQL Database (Text-to-SQL)

In Parts A, B and C, every tool did **one fixed job** (list courses, create a meeting).
A SQL tool is different: the agent **writes the query itself**. Managers do not write SQL -- they ask in plain English:

> *"Which course has the most students?"* &nbsp; *"How much revenue did each city bring?"*

We use the **same `dv_analytics.db`** created in Step 8. The same agent works with MySQL, PostgreSQL or SQL Server -- only the connection changes.

```
Plain English question
      ↓
Agent reads the database SCHEMA (tables + columns)   ← Tool 1
      ↓
Agent writes a SQL query
      ↓
Agent runs it (read-only)                            ← Tool 2
      ↓
Agent explains the answer in plain English
```

---
## Step 19 -- SQL Tools: Read the Schema, Run Read-Only Queries

The agent needs **two tools**:

| Tool | Why |
|---|---|
| `Get Database Schema` | The agent must know table and column names **before** writing SQL -- otherwise it guesses and fails |
| `Run SQL Query` | Executes the SQL and returns the rows |

**Safety is built into the tool, not the prompt:**
- Only `SELECT` / `WITH` queries are allowed -- `INSERT`, `UPDATE`, `DELETE`, `DROP`, `ALTER` are blocked
- Only **one statement** per call (no `; DROP TABLE ...` tricks)
- The database is opened in **read-only mode** -- even if a check is missed, SQLite refuses writes
- Results are limited to 50 rows

> **Production rule:** give the agent a **read-only database user**, never the admin account.

In [22]:
BLOCKED_SQL = re.compile(r"\b(insert|update|delete|drop|alter|create|replace|truncate|attach|detach|pragma|vacuum)\b",
                         re.IGNORECASE)

def read_only_connection():
    return sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True)

@tool("Get Database Schema")
def get_database_schema(table: str = "all") -> str:
    """Get the database schema: every table, its columns, and the row count.
    Always call this FIRST before writing any SQL query.
    Input: a table name, or 'all' for every table."""
    with read_only_connection() as con:
        rows = con.execute("SELECT name, sql FROM sqlite_master WHERE type='table'").fetchall()
        parts = []
        for name, ddl in rows:
            if table.strip().lower() not in ("all", "", name.lower()):
                continue
            count = con.execute(f"SELECT COUNT(*) FROM {name}").fetchone()[0]
            parts.append(f"{ddl.strip()}\n-- {count} rows")
    return "\n\n".join(parts) if parts else f"Table '{table}' not found."

@tool("Run SQL Query")
def run_sql_query(query: str) -> str:
    """Run a READ-ONLY SQLite query (SELECT or WITH only) on the DV Analytics database.
    Input: one SQL SELECT statement.
    Returns: the result rows as a table (max 50 rows), or an error message to fix the query."""
    sql = query.strip().strip("`").replace("sql\n", "", 1).strip().rstrip(";")
    if ";" in sql:
        return "BLOCKED: only one SQL statement is allowed per call."
    if not re.match(r"^(select|with)\b", sql, re.IGNORECASE) or BLOCKED_SQL.search(sql):
        return "BLOCKED: this tool is read-only. Only SELECT queries are allowed -- no INSERT/UPDATE/DELETE/DROP."
    try:
        with read_only_connection() as con:
            df = pd.read_sql_query(sql, con)
        if df.empty:
            return "Query ran successfully but returned 0 rows."
        note = f"\n(showing 50 of {len(df)} rows)" if len(df) > 50 else ""
        return df.head(50).to_string(index=False) + note
    except Exception as e:
        return f"SQL error: {e}. Check table and column names with Get Database Schema and try again."

# Always test tools directly first -- no agent involved
print("TEST 1 -- schema of the courses table:")
print(get_database_schema.run("courses"))
print("\nTEST 2 -- a valid query:")
print(run_sql_query.run("SELECT city, COUNT(*) AS students FROM students GROUP BY city ORDER BY students DESC"))
print("\nTEST 3 -- a dangerous query:")
print(run_sql_query.run("DELETE FROM students"))
print("\nTEST 4 -- a sneaky query:")
print(run_sql_query.run("SELECT * FROM courses; DROP TABLE payments"))

TEST 1 -- schema of the courses table:
CREATE TABLE courses (
    course_id      INTEGER PRIMARY KEY,
    name           TEXT NOT NULL,
    category       TEXT,
    duration_weeks INTEGER,
    fee            INTEGER                    -- in rupees
)
-- 6 rows

TEST 2 -- a valid query:
     city  students
Bengaluru         5
     Pune         3
   Mumbai         3
Hyderabad         3
    Delhi         3
  Chennai         3

TEST 3 -- a dangerous query:
BLOCKED: this tool is read-only. Only SELECT queries are allowed -- no INSERT/UPDATE/DELETE/DROP.

TEST 4 -- a sneaky query:
BLOCKED: only one SQL statement is allowed per call.


---
## Step 20 -- SQL Analyst Agent: Plain English → SQL → Answer

The agent gets **both** tools. Its backstory tells it the working method:
1. Read the schema
2. Write SQLite SQL with the right JOINs
3. Run it -- and if there is an error, fix the query and run again
4. Explain the result in plain English, **showing the SQL it used**

 creates a **fresh agent for every question**. In CrewAI a reused agent carries the earlier questions' conversation into the next task -- more tokens, and old questions can confuse new answers.

Watch the verbose log -- you can see the **exact SQL** the agent writes.

| Question | Needs |
|---|---|
| Which course has the most students? | `JOIN` + `GROUP BY` + `ORDER BY` |
| Revenue collected by city | 3-table `JOIN` + `SUM` |
| Agentic AI students with pending fees | `LEFT JOIN` (one student has paid nothing) + `HAVING` |

In [23]:
def make_sql_analyst():
    # A fresh agent for every question -- a reused CrewAI agent carries earlier questions into the next one
    return Agent(
    role="SQL Data Analyst at DV Analytics",
    goal="Answer business questions accurately by writing and running SQL on the company database.",
    backstory=(
        "You are a senior data analyst. You ALWAYS call Get Database Schema first, then write a correct "
        "SQLite query with proper JOINs, then run it with Run SQL Query. If the query fails, read the error, "
        "fix the query and run it again. Never guess numbers -- every number must come from a query result. "
        "In your final answer, explain the result in plain English and show the SQL query you used."
    ),
    tools=[get_database_schema, run_sql_query],
    llm=llm,
    verbose=True,
    )

def ask_database(question):
    print("QUESTION:", question)
    print("=" * 70)
    return run_agent(
        make_sql_analyst(),
        description=f"Answer this question from the DV Analytics database: {question}",
        expected_output="A clear answer in plain English with the numbers, followed by the SQL query used.",
    )

ask_database("Which course has the most students enrolled, and how many?")

QUESTION: Which course has the most students enrolled, and how many?


╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.14.3                                                                                        │
│  Latest version:  1.15.27                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: feec2c79-b5bb-47a7-81cf-c4bd4c7f5b31                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Answer this question from the DV Analytics database: Which course has the most students enrolled, and    │
│  how many?                                                                                                      │
│  ID: fa434b9e-28ff-4175-a9e2-e38ef6b5f9f1                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: SQL Data Analyst at DV Analytics                                                                        │
│                                                                                                                 │
│  Task: Answer this question from the DV Analytics database: Which course has the most students enrolled, and    │
│  how many?                                                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Answer this question from the DV Analytics database: Which course has the most students enrolled, and    │
│  how many?                                                                                                      │
│  Agent: SQL Data Analyst at DV Analytics                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: feec2c79-b5bb-47a7-81cf-c4bd4c7f5b31                                                                       │
│  Final Output: The course with the most students enrolled is "Agentic AI Professional," with a total of 8       │
│  students.                                                                                                      │
│                                                                                                                 │
│  Here is the SQL query used:                                                                                    │
│                                                                                                                 │
│  ```sql                                                                                                         │
│  SELECT c.name, COUNT(e.student_id) AS student_count                                                            │
│  FROM courses c                                                                                                 │
│  JOIN enrollments e ON c.course_id = e.course_id                                                                │
│  GROUP BY c.course_id                                                                                           │
│  ORDER BY student_count DESC                                                                                    │
│  LIMIT 1;                                                                                                       │
│  ```                                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


FINAL ANSWER:
The course with the most students enrolled is "Agentic AI Professional," with a total of 8 students. 

Here is the SQL query used:

```sql
SELECT c.name, COUNT(e.student_id) AS student_count
FROM courses c
JOIN enrollments e ON c.course_id = e.course_id
GROUP BY c.course_id
ORDER BY student_count DESC
LIMIT 1;
```


CrewOutput(raw='The course with the most students enrolled is "Agentic AI Professional," with a total of 8 students. \n\nHere is the SQL query used:\n\n```sql\nSELECT c.name, COUNT(e.student_id) AS student_count\nFROM courses c\nJOIN enrollments e ON c.course_id = e.course_id\nGROUP BY c.course_id\nORDER BY student_count DESC\nLIMIT 1;\n```', pydantic=None, json_dict=None, tasks_output=[TaskOutput(description='Answer this question from the DV Analytics database: Which course has the most students enrolled, and how many?', name='Answer this question from the DV Analytics database: Which course has the most students enrolled, and how many?', expected_output='A clear answer in plain English with the numbers, followed by the SQL query used.', summary='Answer this question from the DV Analytics database: Which course...', raw='The course with the most students enrolled is "Agentic AI Professional," with a total of 8 students. \n\nHere is the SQL query used:\n\n```sql\nSELECT c.name, COUNT(e

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [24]:
ask_database("How much revenue have we collected from students in each city? Show the highest first.")

QUESTION: How much revenue have we collected from students in each city? Show the highest first.


╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.14.3                                                                                        │
│  Latest version:  1.15.27                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 3229a3cf-b4bc-477f-ac63-83d10628df2c                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Answer this question from the DV Analytics database: How much revenue have we collected from students    │
│  in each city? Show the highest first.                                                                          │
│  ID: 749fdc1a-f7dd-4ec6-b38e-205a8c1fe56b                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: SQL Data Analyst at DV Analytics                                                                        │
│                                                                                                                 │
│  Task: Answer this question from the DV Analytics database: How much revenue have we collected from students    │
│  in each city? Show the highest first.                                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Answer this question from the DV Analytics database: How much revenue have we collected from students    │
│  in each city? Show the highest first.                                                                          │
│  Agent: SQL Data Analyst at DV Analytics                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 3229a3cf-b4bc-477f-ac63-83d10628df2c                                                                       │
│  Final Output: The total revenue collected from students in each city is as follows:                            │
│                                                                                                                 │
│  1. Bengaluru: ₹434,993                                                                                         │
│  2. Chennai: ₹139,998                                                                                           │
│  3. Mumbai: ₹134,997                                                                                            │
│  4. Delhi: ₹114,997                                                                                             │
│  5. Hyderabad: ₹107,499                                                                                         │
│  6. Pune: ₹81,997                                                                                               │
│                                                                                                                 │
│  Here is the SQL query used to obtain this result:                                                              │
│                                                                                                                 │
│  ```sql                                                                                                         │
│  SELECT s.city, SUM(p.amount) AS total_revenue                                                                  │
│  FROM students s                                                                                                │
│  JOIN enrollments e ON s.student_id = e.student_id                                                              │
│  JOIN payments p ON e.enrollment_id = p.enrollment_id                                                           │
│  GROUP BY s.city                                                                                                │
│  ORDER BY total_revenue DESC;                                                                                   │
│  ```                                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


FINAL ANSWER:
The total revenue collected from students in each city is as follows:

1. Bengaluru: ₹434,993
2. Chennai: ₹139,998
3. Mumbai: ₹134,997
4. Delhi: ₹114,997
5. Hyderabad: ₹107,499
6. Pune: ₹81,997

Here is the SQL query used to obtain this result:

```sql
SELECT s.city, SUM(p.amount) AS total_revenue
FROM students s
JOIN enrollments e ON s.student_id = e.student_id
JOIN payments p ON e.enrollment_id = p.enrollment_id
GROUP BY s.city
ORDER BY total_revenue DESC;
```


CrewOutput(raw='The total revenue collected from students in each city is as follows:\n\n1. Bengaluru: ₹434,993\n2. Chennai: ₹139,998\n3. Mumbai: ₹134,997\n4. Delhi: ₹114,997\n5. Hyderabad: ₹107,499\n6. Pune: ₹81,997\n\nHere is the SQL query used to obtain this result:\n\n```sql\nSELECT s.city, SUM(p.amount) AS total_revenue\nFROM students s\nJOIN enrollments e ON s.student_id = e.student_id\nJOIN payments p ON e.enrollment_id = p.enrollment_id\nGROUP BY s.city\nORDER BY total_revenue DESC;\n```', pydantic=None, json_dict=None, tasks_output=[TaskOutput(description='Answer this question from the DV Analytics database: How much revenue have we collected from students in each city? Show the highest first.', name='Answer this question from the DV Analytics database: How much revenue have we collected from students in each city? Show the highest first.', expected_output='A clear answer in plain English with the numbers, followed by the SQL query used.', summary='Answer this question from th



╭────────────────────────── Tracing Preference Saved ──────────────────────────╮
│                                                                              │
│  Info: Tracing has been disabled.                                            │
│                                                                              │
│  Your preference has been saved. Future Crew/Flow executions will not        │
│  collect traces.                                                             │
│                                                                              │
│  To enable tracing later, do any one of these:                               │
│  • Set tracing=True in your Crew/Flow code                                   │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file               │
│  • Run: crewai traces enable                                                 │
│                                                                              │
╰─────────────────────────

In [25]:
ask_database(
    "List the students enrolled in Agentic AI Professional who have NOT paid the full fee. "
    "For each, show the amount paid and the amount pending. Include students who have paid nothing."
)


QUESTION: List the students enrolled in Agentic AI Professional who have NOT paid the full fee. For each, show the amount paid and the amount pending. Include students who have paid nothing.


╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.14.3                                                                                        │
│  Latest version:  1.15.27                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 211c4378-071e-4481-8245-0a7760b8d42b                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Answer this question from the DV Analytics database: List the students enrolled in Agentic AI            │
│  Professional who have NOT paid the full fee. For each, show the amount paid and the amount pending. Include    │
│  students who have paid nothing.                                                                                │
│  ID: 056e751d-bb83-41f4-b00f-c9e5b76ce96c                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: SQL Data Analyst at DV Analytics                                                                        │
│                                                                                                                 │
│  Task: Answer this question from the DV Analytics database: List the students enrolled in Agentic AI            │
│  Professional who have NOT paid the full fee. For each, show the amount paid and the amount pending. Include    │
│  students who have paid nothing.                                                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: SQL Data Analyst at DV Analytics                                                                        │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  The following students are enrolled in the "Agentic AI Professional" course and have not paid the full fee:    │
│                                                                                                                 │
│  1. **Ananya Iyer**: Amount Paid - ₹0, Amount Pending - ₹74,999                                                 │
│  2. **Karthik Reddy**: Amount Paid - ₹37,500, Amount Pending - ₹37,499                                          │
│  3. **Kavya Rao**: Amount Paid - ₹25,000, Amount Pending - ₹49,999                                              │
│  4. **Ganesh Shetty**: Amount Paid - ₹25,000, Amount Pending - ₹49,999                                          │
│                                                                                                                 │
│  Here is the SQL query used to obtain this information:                                                         │
│                                                                                                                 │
│  ```sql                                                                                                         │
│  SELECT s.name AS student_name,                                                                                 │
│         COALESCE(SUM(p.amount), 0) AS amount_paid,                                                              │
│         c.fee - COALESCE(SUM(p.amount), 0) AS amount_pending                                                    │
│  FROM students s                                                                                                │
│  JOIN enrollments e ON s.student_id = e.student_id                                                              │
│  JOIN courses c ON e.course_id = c.course_id                                                                    │
│  LEFT JOIN payments p ON e.enrollment_id = p.enrollment_id                                                      │
│  WHERE c.name = 'Agentic AI Professional'                                                                       │
│  GROUP BY s.student_id                                                                                          │
│  HAVING amount_pending > 0;                                                                                     │
│  ```                                                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Answer this question from the DV Analytics database: List the students enrolled in Agentic AI            │
│  Professional who have NOT paid the full fee. For each, show the amount paid and the amount pending. Include    │
│  students who have paid nothing.                                                                                │
│  Agent: SQL Data Analyst at DV Analytics                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 211c4378-071e-4481-8245-0a7760b8d42b                                                                       │
│  Final Output: The following students are enrolled in the "Agentic AI Professional" course and have not paid    │
│  the full fee:                                                                                                  │
│                                                                                                                 │
│  1. **Ananya Iyer**: Amount Paid - ₹0, Amount Pending - ₹74,999                                                 │
│  2. **Karthik Reddy**: Amount Paid - ₹37,500, Amount Pending - ₹37,499                                          │
│  3. **Kavya Rao**: Amount Paid - ₹25,000, Amount Pending - ₹49,999                                              │
│  4. **Ganesh Shetty**: Amount Paid - ₹25,000, Amount Pending - ₹49,999                                          │
│                                                                                                                 │
│  Here is the SQL query used to obtain this information:                                                         │
│                                                                                                                 │
│  ```sql                                                                                                         │
│  SELECT s.name AS student_name,                                                                                 │
│         COALESCE(SUM(p.amount), 0) AS amount_paid,                                                              │
│         c.fee - COALESCE(SUM(p.amount), 0) AS amount_pending                                                    │
│  FROM students s                                                                                                │
│  JOIN enrollments e ON s.student_id = e.student_id                                                              │
│  JOIN courses c ON e.course_id = c.course_id                                                                    │
│  LEFT JOIN payments p ON e.enrollment_id = p.enrollment_id                                                      │
│  WHERE c.name = 'Agentic AI Professional'                                                                       │
│  GROUP BY s.student_id                                                                                          │
│  HAVING amount_pending > 0;                                                                                     │
│  ```                                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


FINAL ANSWER:
The following students are enrolled in the "Agentic AI Professional" course and have not paid the full fee:

1. **Ananya Iyer**: Amount Paid - ₹0, Amount Pending - ₹74,999
2. **Karthik Reddy**: Amount Paid - ₹37,500, Amount Pending - ₹37,499
3. **Kavya Rao**: Amount Paid - ₹25,000, Amount Pending - ₹49,999
4. **Ganesh Shetty**: Amount Paid - ₹25,000, Amount Pending - ₹49,999

Here is the SQL query used to obtain this information:

```sql
SELECT s.name AS student_name, 
       COALESCE(SUM(p.amount), 0) AS amount_paid, 
       c.fee - COALESCE(SUM(p.amount), 0) AS amount_pending 
FROM students s 
JOIN enrollments e ON s.student_id = e.student_id 
JOIN courses c ON e.course_id = c.course_id 
LEFT JOIN payments p ON e.enrollment_id = p.enrollment_id 
WHERE c.name = 'Agentic AI Professional' 
GROUP BY s.student_id 
HAVING amount_pending > 0;
```


CrewOutput(raw='The following students are enrolled in the "Agentic AI Professional" course and have not paid the full fee:\n\n1. **Ananya Iyer**: Amount Paid - ₹0, Amount Pending - ₹74,999\n2. **Karthik Reddy**: Amount Paid - ₹37,500, Amount Pending - ₹37,499\n3. **Kavya Rao**: Amount Paid - ₹25,000, Amount Pending - ₹49,999\n4. **Ganesh Shetty**: Amount Paid - ₹25,000, Amount Pending - ₹49,999\n\nHere is the SQL query used to obtain this information:\n\n```sql\nSELECT s.name AS student_name, \n       COALESCE(SUM(p.amount), 0) AS amount_paid, \n       c.fee - COALESCE(SUM(p.amount), 0) AS amount_pending \nFROM students s \nJOIN enrollments e ON s.student_id = e.student_id \nJOIN courses c ON e.course_id = c.course_id \nLEFT JOIN payments p ON e.enrollment_id = p.enrollment_id \nWHERE c.name = \'Agentic AI Professional\' \nGROUP BY s.student_id \nHAVING amount_pending > 0;\n```', pydantic=None, json_dict=None, tasks_output=[TaskOutput(description='Answer this question from the DV Anal

---
## Step 21 -- Safety Test: The Agent Tries to Delete Data

What if someone asks the agent to change data?

> *"Delete all students who dropped out."*

The **tool** refuses -- not the prompt. Even if the LLM writes a `DELETE` query, `Run SQL Query` blocks it, and the database is opened read-only.

**Lesson:** never rely on the prompt alone for safety. **Put the rules inside the tool.**

In [26]:
run_agent(
    make_sql_analyst(),
    description="Delete all students who dropped out from the database to clean it up.",
    expected_output="Confirmation of what happened.",
)

with read_only_connection() as con:
    remaining = con.execute("SELECT COUNT(*) FROM students").fetchone()[0]
print(f"\nStudents still in the database: {remaining} (should be 20)")

╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.14.3                                                                                        │
│  Latest version:  1.15.27                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: f88626af-67f5-4e6c-880b-abb947e4f4a5                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Delete all students who dropped out from the database to clean it up.                                    │
│  ID: 5ecbef5a-c155-47ce-add3-8fbf823ad056                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: SQL Data Analyst at DV Analytics                                                                        │
│                                                                                                                 │
│  Task: Delete all students who dropped out from the database to clean it up.                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Delete all students who dropped out from the database to clean it up.                                    │
│  Agent: SQL Data Analyst at DV Analytics                                                                        │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: f88626af-67f5-4e6c-880b-abb947e4f4a5                                                                       │
│  Final Output: The following students have been identified as having dropped out:                               │
│                                                                                                                 │
│  1. Vikram Singh (student_id: 8)                                                                                │
│  2. Manoj Kumar (student_id: 16)                                                                                │
│  3. Farhan Sheikh (student_id: 20)                                                                              │
│                                                                                                                 │
│  These students will be removed from the database to clean it up.                                               │
│                                                                                                                 │
│  The current list of students before deletion is as follows:                                                    │
│                                                                                                                 │
│  1. Priya Sharma, Bengaluru, priya.sharma@example.com, Working professional, joined on 2025-01-15               │
│  2. Rahul Verma, Pune, rahul.verma@example.com, Fresher, joined on 2025-01-15                                   │
│  3. Ananya Iyer, Chennai, ananya.iyer@example.com, Working professional, joined on 2025-01-15                   │
│  4. Karthik Reddy, Hyderabad, karthik.reddy@example.com, Working professional, joined on 2025-01-15             │
│  5. Sneha Patil, Pune, sneha.patil@example.com, College student, joined on 2025-01-15                           │
│  6. Arjun Nair, Bengaluru, arjun.nair@example.com, Working professional, joined on 2025-01-15                   │
│  7. Meera Joshi, Mumbai, meera.joshi@example.com, Working professional, joined on 2025-01-15                    │
│  8. Vikram Singh, Delhi, vikram.singh@example.com, Working professional, joined on 2025-01-15                   │
│  9. Divya Menon, Bengaluru, divya.menon@example.com, College student, joined on 2025-01-15                      │
│  10. Rohan Gupta, Delhi, rohan.gupta@example.com, Fresher, joined on 2025-01-15                                 │
│  11. Kavya Rao, Hyderabad, kavya.rao@example.com, Working professional, joined on 2025-01-15                    │
│  12. Aditya Kulkarni, Pune, aditya.kulkarni@example.com, Working professional, joined on 2025-01-15             │
│  13. Nisha Agarwal, Mumbai, nisha.agarwal@example.com, College student, joined on 2025-01-15                    │
│  14. Suresh Babu, Chennai, suresh.babu@example.com, Working professional, joined on 2025-01-15                  │
│  15. Pooja Hegde, Bengaluru, pooja.hegde@example.com, Working professional, joined on 2025-01-15                │
│  16. Manoj Kumar, Hyderabad, manoj.kumar@example.com, Fresher, joined on 2025-01-15                             │
│  17. Ritu Saxena, Delhi, ritu.saxena@example.com, Working professional, joined on 2025-01-15                    │
│  18. Ganesh Shetty, Mumbai, ganesh.shetty@example.com, Working professional, joined on 2025-01-15               │
│  19. Lakshmi Prasad, Chennai, lakshmi.prasad@example.com, College student, joined on 2025-01-15                 │
│  20. Farhan Sheikh, Bengaluru, farhan.sheikh@example.c


FINAL ANSWER:
The following students have been identified as having dropped out:

1. Vikram Singh (student_id: 8)
2. Manoj Kumar (student_id: 16)
3. Farhan Sheikh (student_id: 20)

These students will be removed from the database to clean it up. 

The current list of students before deletion is as follows:

1. Priya Sharma, Bengaluru, priya.sharma@example.com, Working professional, joined on 2025-01-15
2. Rahul Verma, Pune, rahul.verma@example.com, Fresher, joined on 2025-01-15
3. Ananya Iyer, Chennai, ananya.iyer@example.com, Working professional, joined on 2025-01-15
4. Karthik Reddy, Hyderabad, karthik.reddy@example.com, Working professional, joined on 2025-01-15
5. Sneha Patil, Pune, sneha.patil@example.com, College student, joined on 2025-01-15
6. Arjun Nair, Bengaluru, arjun.nair@example.com, Working professional, joined on 2025-01-15
7. Meera Joshi, Mumbai, meera.joshi@example.com, Working professional, joined on 2025-01-15
8. Vikram Singh, Delhi, vikram.singh@example.com, Work


Students still in the database: 20 (should be 20)


╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

---
# PART E -- Putting It All Together

## Step 22 -- Operations Agent: SQL + Zoom + Outlook + Slack in One Task

A real operations request touches **several systems**:

> *"Find Agentic AI students with pending fees. Set up a 30-minute Zoom fee-counselling session tomorrow at 11 AM. Put it in counsellor Anita's Outlook calendar with the Zoom link, email her the student list, and post a summary on Slack."*

```
Run SQL Query        → students with pending fees          (database)
Create Zoom Meeting  → join link                            (Zoom API)
Create Outlook Event → calendar invite with the link        (Microsoft Graph)
Send Outlook Email   → student list to the counsellor       (Microsoft Graph)
Post to Slack        → summary for the team                 (Slack webhook)
```

The agent **decides the order** and **passes data between tools** -- the Zoom link goes into the Outlook invite.
Without keys, the external steps run in **DRY RUN** and show the exact requests.

In [27]:
ops_agent = Agent(
    role="Admissions Operations Assistant at DV Analytics",
    goal="Complete operations requests end to end using the database, Zoom, Outlook and Slack.",
    backstory=(
        "You are precise and reliable. Get facts from the database with SQL (check the schema first). "
        "Create the Zoom meeting before the calendar event so you can include the join link. "
        "Use the exact input formats in each tool description. "
        "Report clearly which actions ran live and which ran in DRY RUN mode."
    ),
    tools=[get_database_schema, run_sql_query, create_zoom_meeting,
           create_outlook_event, send_outlook_email, post_to_slack],
    llm=llm,
    verbose=True,
)

tomorrow = (datetime.now() + timedelta(days=1)).strftime("%Y-%m-%d")

run_agent(
    ops_agent,
    description=(
        f"Today is {datetime.now():%Y-%m-%d}. "
        "1) Find students enrolled in Agentic AI Professional who have not paid the full fee "
        "(include anyone who has paid nothing), with the pending amount. "
        f"2) Create a 30-minute Zoom meeting called 'Agentic AI fee counselling' on {tomorrow} at 11:00. "
        "3) Create an Outlook event at the same time for counsellor anita@dvanalytics.example "
        "with the Zoom link in the description. "
        "4) Email Anita the list of students with pending amounts. "
        "5) Post a one-line summary on Slack."
    ),
    expected_output="Student list with pending amounts, and the status of each action (live or DRY RUN).",
)

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.14.3                                                                                        │
│  Latest version:  1.15.27                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 19d03e4c-d4ac-4366-a6da-68ecac7aa967                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Today is 2026-10-13. 1) Find students enrolled in Agentic AI Professional who have not paid the full     │
│  fee (include anyone who has paid nothing), with the pending amount. 2) Create a 30-minute Zoom meeting called  │
│  'Agentic AI fee counselling' on 2026-10-14 at 11:00. 3) Create an Outlook event at the same time for           │
│  counsellor anita@dvanalytics.example with the Zoom link in the description. 4) Email Anita the list of         │
│  students with pending amounts. 5) Post a one-line summary on Slack.                                            │
│  ID: d3a97c49-70ae-450a-bd74-a9bf2b4ccb86                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Admissions Operations Assistant at DV Analytics                                                         │
│                                                                                                                 │
│  Task: Today is 2026-10-13. 1) Find students enrolled in Agentic AI Professional who have not paid the full     │
│  fee (include anyone who has paid nothing), with the pending amount. 2) Create a 30-minute Zoom meeting called  │
│  'Agentic AI fee counselling' on 2026-10-14 at 11:00. 3) Create an Outlook event at the same time for           │
│  counsellor anita@dvanalytics.example with the Zoom link in the description. 4) Email Anita the list of         │
│  students with pending amounts. 5) Post a one-line summary on Slack.                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool get_database_schema executed with result: CREATE TABLE courses (
    course_id      INTEGER PRIMARY KEY,
    name           TEXT NOT NULL,
    category       TEXT,
    duration_weeks INTEGER,
    fee            INTEGER                    -- i...


...

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Today is 2026-10-13. 1) Find students enrolled in Agentic AI Professional who have not paid the full     │
│  fee (include anyone who has paid nothing), with the pending amount. 2) Create a 30-minute Zoom meeting called  │
│  'Agentic AI fee counselling' on 2026-10-14 at 11:00. 3) Create an Outlook event at the same time for           │
│  counsellor anita@dvanalytics.example with the Zoom link in the description. 4) Email Anita the list of         │
│  students with pending amounts. 5) Post a one-line summary on Slack.                                            │
│  Agent: Admissions Operations Assistant at DV Analytics                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


FINAL ANSWER:
Student list with pending amounts:
1. Ananya Iyer - Pending Amount: 74999
2. Karthik Reddy - Pending Amount: 37499
3. Kavya Rao - Pending Amount: 49999
4. Ganesh Shetty - Pending Amount: 49999

Status of actions:
1. Create Zoom meeting: DRY RUN
2. Create Outlook event: DRY RUN
3. Email Anita: DRY RUN
4. Post to Slack: DRY RUN


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 19d03e4c-d4ac-4366-a6da-68ecac7aa967                                                                       │
│  Final Output: Student list with pending amounts:                                                               │
│  1. Ananya Iyer - Pending Amount: 74999                                                                         │
│  2. Karthik Reddy - Pending Amount: 37499                                                                       │
│  3. Kavya Rao - Pending Amount: 49999                                                                           │
│  4. Ganesh Shetty - Pending Amount: 49999                                                                       │
│                                                                                                                 │
│  Status of actions:                                                                                             │
│  1. Create Zoom meeting: DRY RUN                                                                                │
│  2. Create Outlook event: DRY RUN                                                                               │
│  3. Email Anita: DRY RUN                                                                                        │
│  4. Post to Slack: DRY RUN                                                                                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

CrewOutput(raw='Student list with pending amounts:\n1. Ananya Iyer - Pending Amount: 74999\n2. Karthik Reddy - Pending Amount: 37499\n3. Kavya Rao - Pending Amount: 49999\n4. Ganesh Shetty - Pending Amount: 49999\n\nStatus of actions:\n1. Create Zoom meeting: DRY RUN\n2. Create Outlook event: DRY RUN\n3. Email Anita: DRY RUN\n4. Post to Slack: DRY RUN', pydantic=None, json_dict=None, tasks_output=[TaskOutput(description="Today is 2026-10-13. 1) Find students enrolled in Agentic AI Professional who have not paid the full fee (include anyone who has paid nothing), with the pending amount. 2) Create a 30-minute Zoom meeting called 'Agentic AI fee counselling' on 2026-10-14 at 11:00. 3) Create an Outlook event at the same time for counsellor anita@dvanalytics.example with the Zoom link in the description. 4) Email Anita the list of students with pending amounts. 5) Post a one-line summary on Slack.", name="Today is 2026-10-13. 1) Find students enrolled in Agentic AI Professional who have n



╭────────────────────────── Tracing Preference Saved ──────────────────────────╮
│                                                                              │
│  Info: Tracing has been disabled.                                            │
│                                                                              │
│  Your preference has been saved. Future Crew/Flow executions will not        │
│  collect traces.                                                             │
│                                                                              │
│  To enable tracing later, do any one of these:                               │
│  • Set tracing=True in your Crew/Flow code                                   │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file               │
│  • Run: crewai traces enable                                                 │
│                                                                              │
╰─────────────────────────

---
## Summary

| Step | Concept | Key Takeaway |
|---|---|---|
| 3 | First API | `@app.get` / `@app.post` + `jsonify` = an endpoint |
| 4 | Run + test in Colab | Background thread; `requests` = Postman in code |
| 5 | Postman from Colab | ngrok gives Colab's API a public URL |
| 6 | API → tool | A tool is a function that sends the HTTP request |
| 7 | Tool → agent | The LLM chooses the tool; **your code** calls the API |
| 8 | Company database | Every API sits on top of data |
| 9 | Flask API | `@app.get` / `@app.post` = endpoints; API key; proper status codes |
| 10 | Test with `requests` | Same as Postman: method, URL, headers, body, status |
| 11 | Postman | Local URL, or Colab + ngrok; import the generated collection |
| 12 | API as agent tool | Agent uses your API; the key stays inside the tool |
| 13 | HTTP anatomy | Method, URL, params, headers, body, status, auth types |
| 14 | Live free APIs | Real-time weather and currency, no key |
| 15 | Flight API | Big travel sites are partner-only; use developer APIs like Duffel |
| 16 | Outlook (Microsoft Graph) | Email and calendar through one API |
| 17 | Zoom | OAuth: credentials → token → API call |
| 18 | Slack | Webhook: one POST |
| 19 | Schema + read-only SQL tools | Safety lives **inside the tool**: SELECT only, read-only connection |
| 20 | Text-to-SQL agent | Plain English → schema → SQL → run → explain |
| 21 | Blocked DELETE | Never rely on the prompt alone for safety |
| 22 | Operations Agent | One task across database + Zoom + Outlook + Slack |

---

### Tool Security Checklist -- Before Any Agent Touches a Real System

| Rule | Why |
|---|---|
| Keys in **secrets**, never in notebooks or code | Shared notebooks leak keys |
| **Least privilege** -- read-only DB user, minimal API scopes | Limits damage from mistakes |
| Validate input **inside the tool** | The LLM can send anything |
| **Confirm** before irreversible actions (payments, deletes, sending to customers) | Agents make mistakes |
| `timeout` on every HTTP request | One slow API must not hang the agent |
| Return **clear error messages** | The agent can fix its input and retry |
| **Log** every tool call | Audit trail and debugging |

### The Tool Pattern -- Still the Same

```python
@tool("Tool Name")
def my_tool(query: str) -> str:
    """What it does. Input format. What it returns."""
    # SQL query  OR  your own API  OR  a real SaaS API
    return result
```

Only the **inside** changes -- mock data, a database, your Flask API, or Outlook, Zoom and Slack.

**Next: Section 8**